<div align="center">
    <h2><b>Architecture Design Document (ADD)</b></h2>
    <h1>Проектування глобальної E-Commerce платформи (Multi-Region)</h1>
    <h3>Глобальний маркетплейс «2BeMarket»</h3>
    <br>

**Метадані документа:**

| **Тип документа:** | High-Level Design (HLD) & Global Scale |
| :--- | :--- |
| **Статус:** | Final / Release |
| **Масштаб:** | Global (Multi-Region Active-Active) |
| **Версія архітектури:** | 3.3.3 |
| **Дата створення:** | Березень 2026 р. |

<br>

**Автор проекту:**

| **Ім'я:** | Oleh Hatsenko (IRONKAGE) |
| :--- | :--- |
| **Роль:** | Principal System Architect / SDET |
| **Організація:** | NeoVersity (Master's in AI & Machine Learning) |
| **GitHub / Contacts:** | github.com/IRONKAGE |

</div>

---

### **Executive Summary (Резюме архітектури):**

Цей документ описує комплексне проектування високорівневої архітектури (HLD) для глобальної платформи електронної комерції **2BeMarket** (аналог Amazon / AliExpress). Платформа розрахована на понад **100 млн DAU** та роботу в умовах екстремальних навантажень (Global Black Friday), витримуючи піки до **175,000 RPS**. Головний архітектурний виклик — забезпечення мілісекундної затримки для покупців по всьому світу зі збереженням суворої консистентності фінансових транзакцій в умовах розподілених систем.

**Ключові архітектурні парадигми та рішення:**
- **Multi-Region Active-Active (4 Хмари):** Розгортання інфраструктури у чотирьох глобальних географічних кластерах (Америка, Європа, Близький Схід/Африка, Азія) на базі AWS, GCP, Azure та Alibaba Cloud за допомогою Global Server Load Balancing (GSLB). Це гарантує виживання системи навіть у разі падіння цілого континенту (Disaster Recovery) та забезпечує суверенітет даних.
- **Quorum Consensus & Split-Brain Protection:** Транзакційне ядро використовує механізми кворуму (Raft/Paxos) та синхронізацію через атомні годинники (UTC) для уникнення розсинхронізації часу (Clock Skew) та захисту фінансів у разі розриву трансокеанічних оптоволоконних кабелів.
- **Conflict-Free Carts (CRDT):** Кошики користувачів реалізовані на базі безконфліктних структур даних (CRDT) у розподіленому NoSQL-сховищі (DynamoDB/Cassandra), що виключає втрату товарів при перемиканні користувача між регіонами, а також між мобільним застосунком та Web.
- **Distributed Transactions & Saga:** Відмова від повільних 2PC-транзакцій на користь патерну **Saga Orchestration** (Оркестрація) з використанням `Idempotency Keys`. Це гарантує надійність списання коштів та інвентаризації складських залишків.
- **Global Polyglot Persistence:** Використання NewSQL (наприклад, Google Spanner / CockroachDB) для транзакційного ядра з суворою консистентністю (ACID) на глобальному рівні, та ElasticSearch для миттєвого повнотекстового пошуку товарів.
- **Machine Learning & Personalization:** Асинхронний пайплайн (Kafka + Flink) для збору клікстріму та віддачі персоналізованих рекомендацій у режимі реального часу як для веб-клієнтів, так і для **мобільних застосунків**.
- **Multi-Cloud & Data Sovereignty:** Інфраструктура є Cloud-Agnostic (Kubernetes) і розгорнута в різних хмарах (AWS для Америки/Східної Азії, GCP для Європи/Швейцарії, Azure для Близького Сходу/Африки, Alibaba для Китаю/Південно-Східної Азії). Це гарантує юридичну відповідність суворим місцевим законам про зберігання персональних даних (GDPR, FADP, PIPL).
- **Edge Security & Geo-Fencing:** Використання Cloudflare WAF на рівні глобального маршрутизатора для захисту від DDoS та жорсткого геоблокування (Geo-Ban) підсанкційних регіонів (наприклад, рф) ще до потрапляння трафіку у внутрішню мережу маркетплейсу.
- **Edge Computing & Virtual Waiting Room:** Використання Cloudflare Workers для відсікання "сміттєвого" трафіку та створення віртуальних черг (Backpressure) під час релізів ексклюзивних товарів, що захищає бекенд від DDoS-атак та ботів.
- **GenAI Core & Shopping Copilot:** Інтеграція RAG-архітектури (Retrieval-Augmented Generation) для розумного пошуку природною мовою у мобільному застосунку та асинхронна LLM-локалізація описів товарів (Dynamic Translation).
- **Full-Stack Observability:** Впровадження розподіленого трасування (OpenTelemetry + Jaeger) для наскрізного контролю кожної транзакції між мікросервісами, базами даних та Kafka.

---

### **Контекстна діаграма потоків (Level 0):**

```mermaid
graph LR
    %% Styling (Luxury & Dark Theme)
    classDef source fill:#2b2b2b,stroke:#00ffcc,stroke-width:2px,color:#fff
    classDef stream fill:#1a1a1a,stroke:#ff3366,stroke-width:2px,color:#fff
    classDef batch fill:#1a1a1a,stroke:#3399ff,stroke-width:2px,color:#fff
    classDef sink fill:#2b2b2b,stroke:#ffcc00,stroke-width:2px,color:#fff
    classDef banned fill:#3a0000,stroke:#ff0000,stroke-width:2px,color:#fff

    %% Нові Люксові Стилі
    classDef buyer_lux fill:#5e35b1,stroke:#ffd700,stroke-width:3px,color:#fff,rx:10px,ry:10px
    classDef seller_lux fill:#00796b,stroke:#ffd700,stroke-width:3px,color:#fff,rx:10px,ry:10px

    Buyer(("📱 Покупці<br>(Мобільний застосунок / Web)")):::buyer_lux
    Seller(("💻 Продавці<br>(Merchant Portal)")):::seller_lux
    Sanctioned(("🚫 Підсанкційний трафік<br>(рф тощо)")):::banned

    WAF{{"🛡️ Edge Security & WAF<br>(Cloudflare / Geo-Ban)"}}:::source
    GLB{{"🌐 Global Load Balancer<br>& API Gateway"}}:::source

    Catalog["🛒 Catalog, Search & AI Copilot<br>(ElasticSearch, Redis, RAG)"]:::batch
    Checkout["💳 Cart & Checkout<br>(Saga Orchestrator)"]:::stream

    Bus[["⚡ Global Event Bus<br>(Apache Kafka)"]]:::source

    Storage[("🗄️ Polyglot Persistence<br>(NewSQL, Cassandra)")]:::sink
    Bank[["🏦 Платіжні Шлюзи<br>(Stripe/PayPal)"]]:::sink
    Logistics[["📦 Логістичні Хаби<br>(DHL/FedEx/Local Post)"]]:::sink

    %% Зв'язки безпеки та роутингу
    Buyer -- "Мільйони запитів/сек<br>(Пошук, Кошик)" --> WAF
    Seller -- "Управління інвентарем" --> WAF
    Sanctioned -. "Connection Dropped" .-> WAF
    WAF -- "Авторизований трафік" --> GLB

    %% Внутрішні потоки
    GLB -- "Read-heavy трафік" --> Catalog
    GLB -- "Write-heavy (Idempotent)" --> Checkout

    Checkout -- "Синхронна оплата" --> Bank
    Checkout -- "Асинхронні події" --> Bus

    Bus -- "Eventual Consistency" --> Storage
    Checkout -. "ACID Транзакції" .-> Storage
    Catalog -. "Індексація даних" .-> Storage
    Bus -- "Push: Інтеграція доставки" --> Logistics
```

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **1. Системні вимоги та Back-of-the-envelope аналіз**

Проект **2BeMarket** створюється як глобальна екосистема, що поєднує покупців та продавців у різних юрисдикціях. Тому вимоги до системи охоплюють не лише технічні, але й суворі юридичні та логістичні аспекти.

### 1.1. Функціональні вимоги (Functional Requirements)

1. **Каталог та Пошук:** Користувачі (**мобільний застосунок** / Web) можуть шукати товари, фільтрувати їх за десятками параметрів та переглядати деталі в режимі реального часу.
2. **Кошик та Оформлення (Checkout):** Можливість додавати товари в кошик, змінювати їх кількість та безшовно переходити до оплати через зовнішні платіжні шлюзи (Stripe, PayPal, Alipay).
3. **Управління замовленнями та Логістика:** Відстеження статусу замовлення, система повернень, а також інтеграція з логістичними хабами (Logistics Gateway) для оновлення трекінг-статусів.
4. **Управління користувачами та Продавцями:** Реєстрація покупців та B2B-портал для продавців (Merchant Portal) з управлінням інвентарем та зонами доставки.
5. **Когнітивна Персоналізація та AI Copilot:** Відмова від базових рекомендацій на користь семантичного векторного пошуку. Інтеграція інтерактивного **Shopping Copilot** на базі LLM/RAG для пошуку природною мовою у **мобільному застосунку**, а також мілісекундна генерація персоналізованих рекомендацій через виділений Feature Store.
6. **Аналітика:** Надання агрегованої статистики продажів для продавців та інвесторів.
7. **Система відгуків та рейтингів (Reviews & Ratings):** Користувачі можуть залишати текстові відгуки, фото та ставити оцінки купленим товарам. Система включає автоматичну ML-модерацію (захист від спаму/ненормативної лексики) та агрегацію середнього рейтингу продавця.
8. **Dynamic Pricing Engine (Динамічне ціноутворення):** Використовуючи патерни з систем бронювання авіаквитків, **2BeMarket** інтегрує ML-базований мікросервіс Pricing Engine. Алгоритм у режимі реального часу аналізує швидкість викупу товарів (Velocity of Sales), історичні дані попиту та поточні залишки на складах (Inventory Yield Management). Це дозволяє застосовувати стратегії динамічного або "surge" ціноутворення для максимізації прибутку без ручного втручання менеджерів.

---

### 1.2. Нефункціональні вимоги (Non-Functional Requirements)

1. **Висока доступність (High Availability):** Цільова доступність **99.99%** (не більше 52 хвилин простою на рік). Це індустріальний стандарт ("Four Nines") для глобальних транзакційних систем. Збереження працездатності гарантується автоматичним Failover-ом навіть при повному падінні цілого хмарного макрорегіону. Вимога "п'яти дев'яток" (99.999% — 5 хвилин на рік) свідомо відкинута як економічно недоцільна, оскільки час DNS-пропагації та перемикання кворуму БД під час міжконтинентального збою може перевищити цей жорсткий ліміт.
2. **Низька затримка (Low Latency):** Час відповіді API \< 200 мс для 99% запитів; повнотекстовий пошук серед мільйонів товарів \< 500 мс у будь-якій точці планети.
3. **Строга консистентність (Strong Consistency):** Гарантія ACID-транзакцій для фінансових операцій та списання залишків зі складу (уникнення проблеми подвійного продажу одного товару).
4. **Data Sovereignty & Комплаєнс:** Суворе дотримання законів про локалізацію персональних даних (GDPR в ЄС, FADP у Швейцарії, PIPL у Китаї). Дані користувачів не повинні перетинати кордони своїх регіонів без дозволу.
5. **Інтернаціоналізація (i18n):** Динамічна підтримка кількох мов, конвертація валют у реальному часі та автоматичний розрахунок регіональних податків (Tax Engine).
6. **Безпека та Відмовостійкість (State-Level Security):** Відповідність стандарту PCI-DSS, захист від DDoS-атак та ботів-скальперів, впровадження парадигми Zero Trust. Додатково архітектура передбачає захист найвищого рівня від інсайдерських та урядових загроз (Confidential Computing, Cryptographic Plausible Deniability) для абсолютної гарантії банківської таємниці та приватності транзакцій.
7. **Обсервабільність (Observability):** Система повинна бути повністю прозорою для дебагінгу розподілених транзакцій (Distributed Tracing), збору метрик у реальному часі та централізованого логування.

---

### 1.3. Оцінка навантаження та ресурсів (Capacity Planning)

Для доведення масштабованості системи наведемо базові (back-of-the-envelope) розрахунки для цільової аудиторії:

- **Трафік:** 100 млн активних користувачів щодня (DAU). При середньому показнику 15 запитів на користувача маємо **1.5 млрд запитів/день**, що дорівнює базовому навантаженню у **\~17,500 RPS** (Requests Per Second).
- **Піковий RPS (Black Friday / Флеш-розпродажі):** Історично трафік у такі дні зростає в 10 разів. Система повинна бути спроектована на витримування **175,000 RPS**. З них приблизно 5% — це транзакції запису (Write), тобто **\~8,750 TPS** (Transactions Per Second), що вимагає потужного брокера повідомлень (Kafka) та оптимізованого пулу підключень до БД.
- **Пропускна здатність (Bandwidth):** Середній розмір відповіді API (JSON-метадані + посилання на CDN) складає близько 50 KB. Базовий вихідний трафік: 17,500 RPS × 50 KB ≈ **875 MB/sec** (близько 7 Gbps на рівні балансувальників).
- **Конкурентні з'єднання (Concurrent Connections / C10M):** Оскільки платформа використовує патерн Virtual Waiting Room та Real-time трекінг замовлень, очікується підтримка до **2-3 мільйонів постійних, одночасних TCP-з'єднань** (через WebSockets/SSE) від клієнтів у пікові моменти. Це вимагає виділеного асинхронного кластера Push Gateway (на базі Go/Rust), щоб не вичерпати пули сокетів на основних транзакційних мікросервісах.
- **Зберігання даних (Storage):** Очікується близько 50 млн нових замовлень на день. Середній розмір об'єкта замовлення (JSON) ≈ 2 KB.
    - *Приріст транзакційної БД:* 50 млн × 2 KB = **100 GB/day**.
    - *За 5 років (з урахуванням 3-кратної реплікації для надійності у Multi-Cloud):* 100 GB × 365 × 5 років × 3 репліки = **~547 TB** високошвидкісного дискового простору (або ~182 TB "чистого" об'єму) лише для ядра транзакцій. Це математично обґрунтовує відмову від монолітного PostgreSQL на користь георозподілених NewSQL рішень (CockroachDB / Spanner).
- **Сховище медіафайлів (Object Storage / BLOBs):** Окрім транзакційного JSON-ядра, платформа генерує колосальний обсяг неструктурованих даних (фото товарів, відео-огляди, картинки у відгуках).
    - *Каталог:* 50 млн товарів × 5 фото/відео × 300 KB = **~75 TB** базового контенту.
    - З урахуванням щоденного завантаження користувацького контенту (UGC) у відгуках та створення різних розмірів зображень (Thumbnails) для CDN, архітектура повинна підтримувати масштабування мультимедійного сховища (AWS S3 / GCP Cloud Storage) до **кількох Петабайтів (PB)** протягом перших 3 років.

**Візуалізація профілю навантаження під час пікових подій:**
```mermaid
pie title ........Розподіл трафіку (Black Friday - 175,000 RPS)
    "Перегляд каталогу та Пошук (Read-heavy)" : 85
    "Генерація рекомендацій (ML Inference)" : 10
    "Кошик та Оформлення (Write-heavy / ACID)" : 5
```

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **2. Високорівнева архітектура (High-Level Design)**

Головний архітектурний виклик **2BeMarket** — це не просто масштабування до 175,000 RPS, а забезпечення суверенітету даних (Data Sovereignty) та уникнення Vendor Lock-in у глобальному масштабі. Для цього система відмовляється від монохмарного підходу на користь гібридної мультихмарної екосистеми.

### 2.1. Глобальна мультихмарна топологія (Multi-Cloud Geo-Strategy)

Щоб мінімізувати затримки (latency), забезпечити юридичну відповідність місцевим законам та уникнути геополітичних ризиків, інфраструктура розгорнута на чотирьох різних платформах:

- **Amazon Web Services (AWS):** Основний бекбон для Північної та Південної Америки, а також для **Східної Азії (Японія, Південна Корея, Тайвань)**. Розгортання в азійських дата-центрах AWS (Tokyo, Seoul) дозволяє уникнути геополітичних ризиків використання китайських провайдерів для цих юрисдикцій, водночас забезпечуючи максимальну ємність.
- **Google Cloud Platform (GCP):** Використовується для Європи (включаючи Швейцарію). GCP обрано через сувору відповідність європейським стандартам безпеки (GDPR, FADP), а також передові інструменти для дата-аналітики.
- **Microsoft Azure:** Покриває регіони з особливими вимогами до корпоративної інфраструктури: **Африка, Австралія, Океанія та Близький Схід (Middle East)**. Використання Azure в ОАЕ та Катарі дозволяє ізолювати транзакційне ядро для інтеграції зі специфічними шлюзами ісламського банкінгу (Sharia Compliant Payments).
- **Alibaba Cloud:** Використовується виключно для **материкового Китаю та країн Південно-Східної Азії** (Індонезія, Малайзія) для дотримання місцевого законодавства (PIPL) та інтеграції з локальними платіжними системами (Alipay, WeChat Pay). Міжхмарний зв'язок (Cross-Cloud) із зовнішнім світом забезпечується через виділені канали Cloud Enterprise Network (CEN).

---

### 2.2. Архітектурна схема глобального роутингу та Federated Analytics

```mermaid
graph TD
    %% Styling (Dark Theme & Luxury)
    classDef source fill:#2b2b2b,stroke:#00ffcc,stroke-width:2px,color:#fff
    classDef mesh fill:#1a1a1a,stroke:#ffcc00,stroke-width:2px,color:#fff
    classDef banned fill:#3a0000,stroke:#ff0000,stroke-width:2px,color:#fff
    classDef dark_circle fill:#0a0a0a,stroke:#ff0000,stroke-width:3px,color:#fff
    classDef lux fill:#5e35b1,stroke:#ffd700,stroke-width:3px,color:#fff,rx:10px,ry:10px

    Client(("📱 Клієнти<br>(Global Traffic)")):::lux
    Sanctioned(("🚫 Підсанкційний трафік<br>(рф тощо)")):::banned

    CF{{"🛡️ Cloudflare<br>Global CDN & WAF (Geo-IP)"}}:::source

    %% Принциповий Geo-Ban
    Sanctioned -.->|Geo-IP Block| CF
    Client -->|HTTPS / GraphQL| CF

    %% Глобальний шар аналітики (Global Data Mesh)
    subgraph Global_Mesh ["Глобальна Мережа Даних / Global Data Mesh (Cloud-Agnostic)"]
        Kafka{"⚡ Apache Kafka<br>(Крос-хмарний MirrorMaker)"}:::mesh
        GlobalLake[("🗄️ Озеро Даних / Global Data Lake<br>(Знеособлені Дані)")]:::mesh
        InvestorDash["📊 Дашборди<br>для Інвесторів"]:::mesh

        Kafka --> GlobalLake --> InvestorDash
    end

    %% --- ФУНДАМЕНТАЛЬНИЙ РІВЕНЬ ХМАРНИХ ПРОВАЙДЕРІВ ---

    subgraph AWS ["🟧&nbsp;Amazon&nbsp;Web&nbsp;Services&nbsp;(AWS)<br>Америка та Сх. Азія (Японія/Корея/Тайвань)"]
        GW_US["API Gateway & BFF"]
        Order_US["Мікросервіси (K8s)"]
        DB_US[("Локальна БД / Local DB")]
        Agg_US["Агрегатор (Spark)"]

        GW_US --> Order_US --> DB_US --> Agg_US
    end

    subgraph GCP ["🟩&nbsp;Google&nbsp;Cloud&nbsp;Platform&nbsp;(GCP)<br>Європа та Швейцарія"]
        GW_EU["API Gateway & BFF"]
        Order_EU["Мікросервіси (K8s)"]
        DB_EU[("Локальна БД<br>(GDPR/FADP Compliant)")]
        Agg_EU["Агрегатор (Spark)"]

        GW_EU --> Order_EU --> DB_EU --> Agg_EU
    end

    subgraph AZURE ["🟦 Microsoft Azure<br>Африка, Австралія, Океанія та Бл. Схід"]
        GW_MEA["API Gateway & BFF"]
        Order_MEA["Мікросервіси (K8s)"]
        DB_MEA[("Локальна БД<br>(Sharia/Local Laws)")]
        Agg_MEA["Агрегатор (Spark)"]

        GW_MEA --> Order_MEA --> DB_MEA --> Agg_MEA
    end

    subgraph ALI ["🟥 Alibaba Cloud<br>Материковий Китай та Півд.-Сх. Азія"]
        GW_AS["API Gateway & BFF"]
        Order_AS["Мікросервіси (K8s)"]
        DB_AS[("Локальна БД<br>(PIPL Compliant)")]
        Agg_AS["Агрегатор (Spark)"]

        GW_AS --> Order_AS --> DB_AS --> Agg_AS
    end

    %% Маршрутизація трафіку від CDN до хмар
    CF -->|Geo-Routing| GW_US
    CF -->|Geo-Routing| GW_EU
	CF -.->|Drop Connection| Blocked(("❌ 403 Forbidden")):::dark_circle
    CF -->|Geo-Routing| GW_MEA
    CF -->|Geo-Routing| GW_AS

    %% 1. Передача знеособленої аналітики (Batch)
    Agg_US -->|Analytics| Kafka
    Agg_EU -->|Analytics| Kafka
    Agg_MEA -->|Analytics| Kafka
    Agg_AS -->|Analytics| Kafka

    %% 2. Миттєва реплікація станів кошиків (CRDT)
    Order_US -.->|CRDT Sync| Kafka
    Order_EU -.->|CRDT Sync| Kafka
    Order_MEA -.->|CRDT Sync| Kafka
    Order_AS -.->|CRDT Sync| Kafka
```

---

### 2.3. Опис ключових глобальних компонентів

1. **Edge Security & Geo-Routing (Cloudflare):** Точка входу в систему. Використовує Anycast-маршрутизацію для направлення клієнта до найближчої хмари. Тут же на рівні CDN реалізовано жорсткий Geo-Ban: трафік з несанкціонованих юрисдикцій просто відкидається (Drop Connection), захищаючи бекенд від "сміттєвого" навантаження.
2. **Cloud-Agnostic Infrastructure (Terraform & Service Mesh):** Управління чотирма різними хмарами вручну призвело б до конфігураційного хаосу. Тому вся інфраструктура описана як код (Infrastructure as Code) за допомогою **Terraform / Crossplane**. Мікросервіси упаковані в Kubernetes, але для згладжування різниці між AWS EKS, Google GKE та Alibaba ACK використовується єдиний глобальний **Service Mesh (наприклад, Cilium або Istio)**. Це створює абстрактний мережевий шар, дозволяючи інженерам розгортати сервіси єдиним пайплайном (ArgoCD), не замислюючись про специфіку базового хмарного провайдера.
3. **API Gateway & BFF (Backend for Frontend):** Одразу за балансувальниками трафік потрапляє на шар API. Щоб не перевантажувати **мобільний застосунок** та Web-клієнти складною оркестрацією мікросервісів, впроваджено патерн BFF. Виділені агрегатори формують оптимізовані відповіді для кожного типу пристроїв, мінімізуючи кількість мережевих запитів (Over-fetching). **При цьому шар BFF є повністю Stateless і не містить жодної бізнес-логіки, що фундаментально унеможливлює розбіжності даних (цін, податків) між веб-версією та смартфоном.**
4. **Data Sovereignty та "Парадокс мандрівника":** Персональні дані клієнтів (PII — імена, імейли, адреси) зберігаються виключно в **Local DB** відповідного регіону і ніколи не перетинають кордони (Compliance by Design). Якщо клієнт подорожує (наприклад, з Європи до США) і відкриває **мобільний застосунок**, транскордонна синхронізація його сесії та кошика оперує *виключно знеособленими криптографічними ідентифікаторами (UUID) та товарними SKU*. Передача цих знеособлених станів (через механізм CRDT) між хмарами відбувається через ізольовані системні топіки Kafka по приватних магістралях. Це геніально вирішує "парадокс мандрівника": платформа зберігає єдиний глобальний User Experience, не порушуючи при цьому жорстких вимог щодо локалізації даних (GDPR, FADP).
5. **Federated Analytics, State Replication & Private Backbone:** Крос-хмарна комунікація між AWS, GCP, Azure та Alibaba відбувається НЕ через публічний інтернет, а виключно через виділені оптичні магістралі **Cloud Interconnect (наприклад, Equinix Fabric / Megaport)**. Цей глобальний приватний бекбон (на базі Kafka MirrorMaker) ізолює трафік на рівнях L2/L3 і обслуговує два критичні потоки даних (Data Streams), жоден з яких не містить PII:
    - *Потік 1 (State Replication):* Миттєва транскордонна реплікація знеособлених станів користувача (PII-stripped). Саме цим каналом передаються вектори CRDT-кошиків та криптографічні сесії (UUID) "мандрівників" (з Пункту 4), дозволяючи зберігати глобальний консистентний User Experience без порушення локалізації даних.
    - *Потік 2 (Federated Analytics):* Передача агрегованих бізнес-метрик. Локальні агрегатори (Apache Spark) обробляють сирі дані всередині кожної хмари, повністю видаляють імена та адреси, і генерують суху статистику (наприклад, "продано 10,000 смартфонів у Цюриху"). Для оптимізації Egress-трафіку (FinOps) ці дані жорстко батчаться і пакуються у колонкові формати (Parquet із компресією Snappy) перед відправкою у Global Data Lake.
6. **AI/ML Екосистема та Когнітивний Пошук (GenAI Core):** Оскільки **2BeMarket** об'єднує продавців і покупців з різних мовних середовищ, платформа використовує асинхронний AI-пайплайн (Kafka + LLM). Коли продавець створює товар, система автоматично генерує та кешує семантично правильні переклади описів цільовими мовами регіонів. Крім того, на зміну класичному лексичному пошуку (ElasticSearch) у **мобільний застосунок** інтегровано Shopping Copilot на базі RAG (Retrieval-Augmented Generation). Це дозволяє покупцям здійснювати пошук природною мовою (наприклад: *"знайди теплі зимові черевики, не шкіряні, до 150 франків"*). Базовий стек розширено Векторною БД (Milvus / Qdrant) для семантичного пошуку та Feature Store для мілісекундної генерації персоналізованих рекомендацій.
7. **Глобальний шар Ідентифікації (Global CIAM) та Захист від SPOF:** Оскільки інфраструктура розкидана по 4-х макрорегіонах, платформа використовує єдиний Federated Identity Provider (наприклад, Auth0 / PingIdentity). Щоб уникнути створення Єдиної точки відмови (Single Point of Failure), система покладається на **Stateless-автентифікацію**. При логіні користувача генерується асиметрично підписаний JWT-токен. Усі регіональні API Gateways агресивно кешують публічні ключі (JWKS). Це дозволяє кластерам AWS або GCP миттєво верифікувати токени користувачів локально, без мережевих запитів до провайдера. Навіть якщо глобальний Auth0 повністю впаде, мільйони вже авторизованих користувачів зможуть продовжувати покупки, що гарантує виживання платформи під час інфраструктурних колапсів третіх сторін.

---

### 2.4. Зведена таблиця технологічного стеку (Tech Stack Justification)

Для реалізації глобальної архітектури було обрано збалансований стек, що поєднує перевірені часом стандарти ("Boring Tech") з інноваційними рішеннями для специфічних вузьких місць ("Dark Horses"). Кожне рішення базується на строгих вимогах щодо масштабованості, консистентності (CAP-теорема) та відмовостійкості платформи.

| **Категорія** | **Компонент системи** | **Обрана технологія / Патерн** | **Альтернативи (Відхилено)** | **Обґрунтування (Trade-offs & Justification)** |
| :--- | :--- | :--- | :--- | :--- |
| **Edge Security & Доступність** | Edge Routing & WAF | **Cloudflare** (WAF + Workers) | AWS CloudFront, Fastly | **(Industry Standard):** Найкращий захист від L7-DDoS. Edge Workers дозволяють реалізувати інноваційний "Virtual Waiting Room", відсікаючи сміттєвий трафік ще до бекенду. |
| **Fault Tolerance & Балансування** | Global Load Balancer | **GCP Cloud Load Balancing** | AWS Route 53 | **(Boring Tech):** Надійне та "нудне" рішення. Підтримує Anycast IP (одна адреса для всього світу), забезпечуючи мінімальну затримку маршрутизації між континентами. |
| **Тип сховища (SQL)** | Транзакційне ядро | **CockroachDB** (Distributed NewSQL) | PostgreSQL, AWS Aurora | **(Dark Horse):** Замість класичного PostgreSQL (який важко глобально шардувати), цей NewSQL дає трансконтинентальні ACID-гарантії. *Trade-off:* Висока складність (алгоритм Raft), але абсолютний захист від Split-Brain. |
| **Тип сховища (NoSQL)** | Каталог товарів (Пошук) | **ElasticSearch** (Search) + **MongoDB** (Document) | Solr, Couchbase, Cassandra | **(Proven Tech):** MongoDB ідеальна для збереження динамічних JSON-атрибутів, а ElasticSearch — перевірений еталон для мілісекундного пошуку по 50+ млн товарів із підтримкою складної фасетної фільтрації. |
| **Стратегія кешування** | Кошик (In-Memory) | **Redis Enterprise** (з CRDT) | Amazon DynamoDB, Memcached, Hazelcast | **(Dark Horse):** Замість простого Key-Value кешу використана елітна математична модель CRDT. Вона гарантує безконфліктне злиття кошиків з мобільного застосунку та Web навіть при тимчасовій втраті зв'язку між дата-центрами. |
| **Обробка черг і подій** | Оркестратор Саги (Events) | **Apache Kafka** (Event Streaming) | RabbitMQ, AWS SQS | **(Modern Standard):** Де-факто стандарт для Event-Driven архітектур. Забезпечує патерн Saga Orchestration для розподілених транзакцій (Checkout) та гарантує надійний Event Sourcing при збоях. |
| **Масштабування (Compute)** | Обчислювальний шар | **Kubernetes** (EKS / GKE) + Горизонтальне (HPA) | Вертикальне (Scale-up), AWS ECS | **(Modern Standard):** Для 10 млн DAU це єдиний шлях до стратегії Vendor Exit та Cell-based архітектури. K8s дозволяє миттєво додавати ноди під час пікових розпродажів (Hype Drops) і зменшувати їх для економії (FinOps). |
| **Сховище файлів (Blob)** | Сховище медіа (Object) | **Amazon S3** (Standard + WORM) | GCP Cloud Storage | **(Boring Tech):** Золотий стандарт індустрії. Патерн WORM (Write Once Read Many) забезпечує незмінність аудит-логів для комплаєнсу та безпеки (The Black Book). |
| **Моніторинг та логування** | Observability & Tracing | **OpenTelemetry + Jaeger** | Datadog (повністю) | **(Rising Trend):** Відмова від дорогого вендор-локу на користь відкритого стандарту. Дозволяє застосовувати Tail-based Sampling та економити до 90% інфраструктурного бюджету на зберіганні логів. |
| **Комунікація сервісів** | Внутрішнє / Зовнішнє API | **gRPC** + **GraphQL/REST** (Зовнішня - BFF) | Тільки REST (JSON) | **(Modern Hype):** GraphQL рятує застосунки від Over-fetching (завантаження зайвих даних). Внутрішня комунікація через gRPC (Protobuf) швидша за REST у 7-10 разів і споживає менше ресурсів CPU. |
| **Zero Trust & Security** | Апаратна ізоляція (Data in Use) | **AWS Nitro Enclaves / eBPF** | Класичні VM, AWS GuardDuty | **(Elite Dark Horse):** Рівень захисту FinTech та спецслужб. Апаратне шифрування RAM та перехоплення викликів ядра (Ghost Sharding). *Trade-off:* екстремально високий поріг входу для інженерів платформи. |

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **3. Дизайн даних (Polyglot Persistence) та API**

У глобальній системі з навантаженням до 175,000 RPS використання єдиної монолітної реляційної бази даних є архітектурним антипатерном (Single Point of Failure та вузьке місце масштабування). Тому **2BeMarket** використовує підхід **Polyglot Persistence** — застосування різних спеціалізованих сховищ даних для різних бізнес-доменів.

### 3.1. Вибір баз даних та стратегії зберігання

Кожен регіональний кластер (AWS, GCP тощо) містить власну ізольовану комбінацію сховищ:

1. **Транзакційне ядро (Orders & Payments): NewSQL (CockroachDB / Google Spanner)**
    - *Чому:* Нам потрібні суворі ACID-транзакції для фінансів, але з можливістю горизонтального масштабування. Для вирішення проблеми відносності часу в розподілених системах (Clock Skew) та уникнення транзакційних конфліктів між дата-центрами в різних півкулях, транзакційні бази покладаються на строгу синхронізацію через атомні годинники та GPS (наприклад, TrueTime API у Spanner).
    - *Стратегія:* Абсолютно всі події та фінансові транзакції платформи **2BeMarket** фіксуються виключно в **UTC (Coordinated Universal Time)**. Конвертація в локальні часові пояси з урахуванням літнього/зимового часу (DST) та регіональних зміщень відбувається виключно на рівні презентації (**мобільний застосунок** або веб-браузер клієнта). Це унеможливлює логічні "подорожі в часі" при розрахунку тривалості сесій чи термінів доставки.
2. **Каталог та Пошук (Product Catalog): ElasticSearch + Redis + Vector DB**
    - *Чому:* 85% трафіку — це пошук товарів. ElasticSearch забезпечує миттєвий повнотекстовий та фасетний пошук, а кластер Redis діє як Distributed Cache для найпопулярніших товарів (стратегія *Cache-Aside*). Додатково розгорнута Векторна БД (Milvus / Qdrant) для семантичного пошуку та обробки AI-запитів від Shopping Copilot.
    - **ML Inference та Feature Store:** Оскільки швидкість видачі персоналізованих рекомендацій є критичною для конверсії, система не розраховує ML-ознаки (features) "на льоту" під час кожного кліку. Натомість розгорнуто **Feature Store** (наприклад, Feast). Сервіс рекомендацій миттєво (< 10 мс) отримує з нього попередньо розраховані актуальні ознаки користувача та товару для точного ранжування видачі, забезпечуючи повноцінний семантичний пошук та пошук за зображеннями без навантаження на основні бази даних.
3. **Кошики користувачів (Shopping Carts): NoSQL Wide-Column (Apache Cassandra)**
    - *Чому:* Кошики генерують гігантську кількість операцій запису (Write-heavy). Cassandra ідеально масштабується на запис.
    - *Вирішення конфліктів стану кошика (CRDT):* Для забезпечення безконфліктного злиття кошиків при перемиканні користувача між **мобільним застосунком** та вебом застосовується концепція CRDT. Оскільки Apache Cassandra нативно не підтримує складні структури CRDT "з коробки", логіка вирішення конфліктів (Merge Logic) імплементована на рівні Application Layer. Cart Service зчитує вектори версій з Cassandra, детерміновано виконує злиття станів у пам'яті мікросервісу та записує фінальний консистентний стан назад у БД.
4. **Сховище медіа (Images & Videos): Object Storage (S3 / GCS)**
    - *Чому:* Зберігання петабайтів фотографій товарів та відео-оглядів. Доступ до них здійснюється виключно через глобальну CDN (Cloudflare).
5. **Система відгуків та рейтингів: Document Store (MongoDB / Amazon DocumentDB)**
    - *Чому:* Відгуки мають неструктурований характер (можуть містити текст різної довжини, масиви посилань на фото/відео, вкладені коментарі). Документо-орієнтована NoSQL база дозволяє гнучко зберігати такі дані без жорсткої схеми, забезпечуючи швидке читання разом із завантаженням картки товару.

---

### 3.2. Схема даних (ER Diagram)

Нижче наведено концептуальну модель даних (Entity-Relationship) для ключових сутностей транзакційного ядра.

```mermaid
erDiagram
    %% Entities
    USER {
        uuid user_id PK
        string email
        string password_hash
        string region "EU, US, AS, MEA"
        datetime created_at
    }

    CART_ITEM {
        uuid cart_id FK
        uuid product_id FK
        int quantity
    }

    PRODUCT {
        uuid product_id PK
        string seller_id FK
        string name
        int price_cents "Ціна в центах/рапенах/копійках"
        string currency "ISO: USD, CHF, EUR"
        string status "ACTIVE, OUT_OF_STOCK, BANNED"
    }

	INVENTORY_LEDGER {
        uuid inventory_id PK
        uuid product_id FK
        string region_id "EU_CENTRAL, US_EAST"
        int available_qty
        int reserved_qty "Блокування під час Checkout"
    }

    CART {
        uuid cart_id PK
        uuid user_id FK
        datetime updated_at
        string status "OPEN, CHECKED_OUT"
    }

    ORDER {
        uuid order_id PK
        uuid user_id FK
        int total_amount_cents
        string currency
        string status "PENDING, PAID, FAILED, REFUNDED, SHIPPED"
        string idempotency_key "UK"
        datetime created_at
    }

    ORDER_ITEM {
        uuid order_id FK
        uuid product_id FK
        int quantity
        int locked_price_cents "Фіксація ціни в центах"
    }

    %% Relationships
    USER ||--o{ ORDER : "places"
    USER ||--|| CART : "owns"
    PRODUCT ||--o{ CART_ITEM : "added as"
    CART ||--o{ CART_ITEM : "contains"
    ORDER ||--|{ ORDER_ITEM : "includes"
    PRODUCT ||--o{ ORDER_ITEM : "ordered as"
	PRODUCT ||--o{ INVENTORY_LEDGER : "stocked in"
```

---

### 3.3. Дизайн API (Ключові контракти)

Для взаємодії між **мобільним застосунком** / веб-клієнтом та API Gateway використовується RESTful підхід (з елементами GraphQL для складних запитів каталогу). Нижче наведено 4 критично важливі ендпоінти платформи.

1. **Пошук товарів (Read-Heavy API):**
    - **Ендпоінт:** `GET /api/v1/products/search`
    - **Джерело даних:** ElasticSearch + Redis Cache + Vector DB + Feature Store (Feast)
    - **Параметри запиту:** `?q=laptop&category=electronics&sort=price_asc&limit=20&page=1`
    - **Відповідь (200 OK):** Повертає масив об'єктів товарів. Затримка \< 100мс.
2. **Додавання товару в кошик (Write-Heavy API):**
    - **Ендпоінт:** `POST /api/v1/cart/items`
    - **Джерело даних:** Apache Cassandra
    - **Тіло запиту (Payload):**
	```json
		{
		  "product_id": "uuid-1234",
		  "quantity": 1
		}
	```
    - **Особливість:** Операція є ідемпотентною (патерн *Upsert*). Якщо клієнт відправить той самий запит двічі через збій мережі, кількість товарів не подвоїться помилково, а перезапишеться до актуального стану.
3. **Оформлення замовлення та Оплата (Transactional API):**
    - **Ендпоінт:** `POST /api/v1/orders/checkout`
    - **Джерело даних:** CockroachDB (Запускає Saga Pattern)
    - **Заголовки (Headers):** `Idempotency-Key: <unique_uuid>` *(Критично важливо для запобігання подвійному списанню коштів)*
    - **Тіло запиту:**
	```json
		{
		  "cart_id": "uuid-5678",
		  "payment_method_id": "pm_9876",
		  "shipping_address_id": "addr_123"
		}
	```
    - **Ідемпотентність API на рівні Gateway:** Для захисту фінансового ядра **2BeMarket** від дублюючих транзакцій під час нестабільного мережевого з'єднання (наприклад, коли **мобільний застосунок** виконує автоматичні Retry-запити), впроваджено жорсткий патерн ідемпотентності. Кожен мутаційний запит супроводжується унікальним UUID (`Idempotency-Key`). API Gateway кешує ці ключі в Redis (з TTL 24 години). У разі повторного отримання запиту з тим самим ключем, система повертає закешовану відповідь попередньої успішної операції, гарантуючи, що подвійного списання коштів не відбудеться.
    - **Відповідь (202 Accepted):** Оскільки транзакція розподілена, API не чекає завершення платежу. Воно повертає `order_id` зі статусом `PENDING`, а подальша обробка йде асинхронно.
4. **Перевірка статусу замовлення (Async Polling / SSE):**
    - **Ендпоінт:** `GET /api/v1/orders/{order_id}/status`
    - **Опис:** Використовується **мобільним застосунком** для періодичного опитування (Polling) або через WebSocket / Server-Sent Events (SSE) для отримання Push-оновлень статусу в режимі реального часу після виклику Checkout (наприклад, перехід від `PENDING` до `PAID`).

---

### 3.4. Обробка граничних випадків та Anti-Fraud логіка (Edge Cases)

У глобальному середовищі поведінка користувачів є непередбачуваною. Архітектура **2BeMarket** закладає стійкість до наступних сценаріїв:

1. **Cross-Device Synchronization (Web vs. Mobile):** Кошики користувачів прив'язані до `user_id` у розподіленій базі Cassandra. Використання патерну CRDT (Conflict-free Replicated Data Types) гарантує, що якщо покупець додасть товар з **мобільного застосунку**, а потім продовжить покупки через Web-інтерфейс, стани кошиків безконфліктно об'єднаються без втрати даних, незалежно від порядку надходження запитів.
2. **Cross-Region Hop (Використання VPN) та Global Inventory:** Якщо користувач змінює гео-локацію (наприклад, вмикає VPN США) перед самим оформленням замовлення, Global Load Balancer перенаправить його трафік у новий хмарний регіон (наприклад, з GCP до AWS). Глобальна реплікація баз даних та єдиний шар Global CIAM гарантують наявність кошика та авторизації в новому регіоні. Проте перед ініціалізацією оплати система примусово виконує два кроки:
    - Викликає `Tax Engine` для перерахунку податків та цін згідно з новою локацією.
    - Викликає `Inventory Allocation Service`, щоб перевірити наявність товарів з кошика на локальних складах нового регіону. Якщо товар недоступний локально (наприклад, є лише на європейському складі), користувач отримує попередження в **мобільному застосунку** про збільшену вартість та термін міжнародної доставки, або пропозицію замінити товар на локальний аналог.
3. **Розділення Billing та Shipping Address (Сценарій "Подарунок"):** Система не робить жорсткої прив'язки методів оплати до адреси доставки. Це дозволяє здійснювати транскордонні покупки (наприклад, оплата швейцарською карткою для доставки в іншу країну).
    - *Compliance:* Митні правила та податки розраховуються суворо за **Shipping Address** (адресою доставки).
    - *Anti-Fraud:* Розбіжність між Geo-IP, Billing Address та Shipping Address не призводить до автоматичного блокування. Замість цього транзакція отримує підвищений Risk Score, що тригерить додаткову верифікацію клієнта через **3D Secure** на боці платіжного шлюзу (Stripe / PayPal).
4. **Застарівання ціни (Price Staleness) та Захист від підміни:** Оскільки товари можуть знаходитися в кошику (Cassandra) днями, ціна або валідність промокодів може змінитися (особливо з огляду на роботу `Dynamic Pricing Engine`). Для захисту від цього, клієнтські застосунки ніколи не передають фінальну суму до оплати в тілі POST-запиту Checkout (Zero Trust до клієнта). Замість цього API Gateway викликає перерахунок (Price Snapshot) на основі актуальних даних з `Catalog Service`. Якщо ціна змінилася порівняно з тією, що була закешована у клієнта на екрані, бекенд перериває операцію і повертає статус `409 Conflict` із новою ціною, вимагаючи від користувача явного підтвердження оновленого кошика (Accept New Price) перед ініціалізацією Saga-оркестратора.
5. **Dynamic Pricing Engine (Динамічне ціноутворення):** Використовуючи патерни з систем бронювання авіаквитків (Surge Pricing), **2BeMarket** інтегрує ML-базований мікросервіс ціноутворення. Цей алгоритм у режимі реального часу аналізує **Velocity of Sales** (швидкість викупу конкретного товару), історичні дані попиту та поточні залишки на складах (**Inventory Yield Management**). Це дозволяє системі автоматично підвищувати маржинальність на дефіцитні товари або застосовувати мікро-знижки для розпродажу залишків, максимізуючи прибуток (Revenue) без ручного втручання категорійних менеджерів.

---

### 3.5. Патерн Backend for Frontend (BFF) та Агрегація даних

Хоча глобальний API Gateway є єдиною точкою входу (Single Point of Entry), архітектура **2BeMarket** враховує кардинально різні потреби клієнтських платформ. Замість того, щоб змушувати клієнтів самостійно оркеструвати виклики до десятків мікросервісів (Over-fetching або Under-fetching), між Gateway та транзакційним ядром розгорнуто шар **BFF (Backend for Frontend)**:

- **Mobile BFF:** Оптимізований агрегатор для користувачів смартфонів. Оскільки **мобільний застосунок** працює в умовах нестабільного зв'язку (3G/LTE) та обмеженого заряду батареї, Mobile BFF бере на себе важку роботу. Він паралельно опитує `Catalog Service`, `Review Service` та `Inventory Service`, агрегує відповіді, відкидає зайві метадані та повертає на смартфон єдиний, максимально легкий JSON. Це зменшує кількість мережевих запитів (Round-trips) з 5-7 до 1, радикально прискорюючи рендер екранів.
- **Web BFF:** Агрегатор для десктопного порталу. Він повертає важчі масиви даних (наприклад, зображення високої роздільної здатності, розгорнуті таблиці характеристик та складні аналітичні віджети для продавців), використовуючи потужні обчислювальні ресурси браузера та стабільне широкосмугове з'єднання клієнта.

> ⚖️ **Архітектурне правило "Dumb BFF" (Захист від протиріч):** Наявність двох різних агрегаторів створює ризик різного відображення ціни кошика на комп'ютері та у смартфоні. Щоб унеможливити протиріччя, архітектура жорстко забороняє розміщувати бізнес-логіку на рівні BFF. Вони є "тупими" трубами для перепакування даних. Усі розрахунки знижок та податків (`Tax Engine`) виконуються виключно доменними мікросервісами, гарантуючи ідентичні цифри для вебу та **мобільного застосунку**.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **4. Розподілені транзакції (Saga Pattern) та Надійність**

У монолітних системах фінансова консистентність (наприклад, списання коштів та зменшення залишків на складі) гарантується базою даних через механізм **2PC (Two-Phase Commit)**. Проте в глобальному проекті **2BeMarket** із мікросервісною архітектурою та десятками тисяч запитів на секунду використання 2PC є неможливим через розподілені блокування (Distributed Locks), які знищать пропускну здатність бази під час розпродажів.

Рішенням є використання патерну **Saga (Orchestration)**, де глобальна транзакція розбивається на серію локальних, а у разі збою на будь-якому етапі запускаються **компенсуючі транзакції (Compensating Transactions)** для відкату системи до початкового стану.

### 4.1. Вибір типу Saga: Orchestration vs Choreography

Для процесу Checkout обрано **Saga Orchestration**.

- *Чому не Choreography:* У хореографії мікросервіси реагують на події один одного без єдиного центру. Для фінансових потоків це створює "Spaghetti-архітектуру", яку неможливо дебажити.
- *Наш вибір:* **Order Service** виступає як "Оркестратор". Він є центральним мозком (State Machine), який чітко знає, на якому етапі знаходиться замовлення, і віддає команди (Commands) іншим сервісам через брокер повідомлень Apache Kafka.

---

### 4.2. Sequence Діаграма процесу Checkout (Saga Happy Path & Rollback)

На діаграмі нижче показано асинхронний флоу оформлення замовлення, включаючи взаємодію з клієнтом (веб-порталом та **мобільним застосунком**) та відкат транзакції у разі нестачі коштів на картці. Зверніть увагу на абстракцію платіжного рівня (Payment Router).

```mermaid
sequenceDiagram
    autonumber
    actor Client as 📱 Клієнт (Web/Mobile)
    participant GW as 🌐 API Gateway
    participant Push as 📡 Push Gateway (WebSockets/C10M)
    participant Saga as 📦 Order Service (Orchestrator)
    participant Kafka as ⚡ Apache Kafka
    participant Inv as 🛒 Inventory Service
    participant Pay as 💳 Payment Router

    Client->>GW: POST /checkout (Idempotency-Key)
    GW->>Saga: Ініціалізація замовлення (PENDING)
    Saga-->>Client: 202 Accepted (Повертає order_id)

    Note over Saga, Pay: --- Початок Saga Orchestration ---

    Saga->>Kafka: Publish: [Reserve_Inventory_Cmd]
    Kafka->>Inv: Consume: Command

    alt Залишки присутні (Success)
        Inv->>Kafka: Publish: [Inventory_Reserved_Event] (Semantic Lock / 15m TTL)
		Kafka->>Saga: Consume: [Inventory_Reserved_Event]
        Saga->>Kafka: Publish: [Process_Payment_Cmd]
        Kafka->>Pay: Consume: Command
        Pay->>Stripe/Alipay: Async API Call (Webhook очікування)

        alt Оплата успішна
            Pay-->>Kafka: Publish: [Payment_Success_Event]
            Kafka->>Saga: Consume: Event
            Saga->>Kafka: Publish: [Commit_Inventory_Cmd]
            Kafka->>Inv: Consume: Command

            Note right of Kafka: 🔄 Асинхронний Push
            Kafka->>Push: Consume: [Order_Paid_Event]
            Push-->>Client: WebSocket: Статус PAID ✅
        else Оплата відхилена
            Pay-->>Kafka: Publish: [Payment_Failed_Event]
            Kafka->>Saga: Consume: Event
            Saga->>Kafka: Publish: [Release_Inventory_Cmd] (Розблокування)
            Kafka->>Inv: Consume: Command

            Kafka->>Push: Consume: [Order_Failed_Event]
            Push-->>Client: WebSocket: Статус FAILED ❌
        end

    else Немає на складі (Out of Stock)
        Inv->>Kafka: Publish: [Inventory_Failed_Event]
        Kafka->>Saga: Consume: Event
		Saga->>Saga: DB Update: Order Status -> CANCELED
        Kafka->>Push: Consume: [Inventory_Failed_Event]
        Push-->>Client: WebSocket: Статус CANCELED ❌
    end
```

---

### 4.3. Ключові механізми надійності в Saga

Щоб цей флоу не зламався при падінні мережі або перезавантаженні серверів, проект використовує три залізних правила:

1. **Ідемпотентність Оркестратора:** Усі повідомлення, що проходять через Kafka, містять унікальний `trace_id`. Якщо Kafka двічі надішле подію `[Inventory_Reserved_Event]` через мережевий збій, Order Service перевірить свій внутрішній стан і проігнорує дублікат.
2. **Паттерн Transactional Outbox:** Оркестратор не може просто записати статус у базу і надіслати подію в Kafka одночасно (це вимагало б 2PC між БД та Kafka). Замість цього статус замовлення та сама подія записуються в *одну таблицю БД* в межах однієї ACID-транзакції. Окремий процес (наприклад, Debezium / Change Data Capture) читає цю таблицю і гарантовано пересилає подію в Kafka (At-Least-Once Delivery).
3. **Патерн Semantic Lock та Ізоляція ресурсів (TTL):** Оскільки розподілені транзакції (Saga) не підтримують сувору ACID-ізоляцію на рівні всієї екосистеми, виникають два критичні ризики: технічне "зависання" залишків та бізнес-аномалія "Dirty Read" (Брудне читання). Щоб вирішити це, `Inventory Service` (Крок 5 на схемі) не видаляє товар одразу, а переводить його у колонку `reserved_qty` в таблиці `INVENTORY_LEDGER` на 15 хвилин (TTL). Це елегантно закриває обидві проблеми:
    - *Технічна стійкість:* Якщо система падає і подія про фінальну оплату або скасування губиться назавжди (наприклад, через критичний збій мережі), після закінчення TTL товар автоматично повертається у вільний продаж. Це унеможливлює мертві блокування ("зависання") складських залишків.
    - *Захист конверсії (Бізнес-UX):* Уявіть, що клієнт А ініціює покупку останнього iPhone, а клієнт Б дивиться каталог, доки банк перевіряє картку клієнта А. Завдяки семантичному блокуванню, для клієнта Б цей товар відображається у веб-порталі або **мобільному застосунку** як "У кошику іншого покупця". Якщо оплата клієнта А відхиляється шлюзом, товар автоматично повертається у вільний продаж, і клієнт Б отримує Push-сповіщення, що товар знову доступний. Це захищає бізнес від втрати реального покупця через тимчасові проміжні стани транзакцій.
4. **State Machine Persistence (Стійкість Оркестратора):** Оскільки мікросервіси в Kubernetes є ефемерними (Pod може бути знищений у будь-яку секунду), стан самої Saga-транзакції ніколи не зберігається лише в оперативній пам'яті `Order Service`. Для оркестрації використовується стійкий Event-Sourced рушій (наприклад, **Temporal.io** або спеціалізована таблиця `saga_state` у CockroachDB). Це гарантує, що якщо вузол Оркестратора критично завершує роботу посеред оформлення замовлення, інший доступний вузол миттєво "прокидається", зчитує останній підтверджений стан із бази і продовжує оркестрацію транзакції або ініціює її відкат (Rollback) рівно з тієї мілісекунди, де стався збій.
5. **Гарантія послідовності (Kafka Partition Key):** У розподілених системах події можуть надходити не по порядку (Race Conditions) через затримки мережі або Retry-механізми. Щоб подія про успішну оплату ніколи не обігнала подію про резервацію, абсолютно всі повідомлення, пов'язані з одним замовленням, публікуються в Kafka з використанням `order_id` як Partition Key. Це гарантує строгу послідовність (Strict Ordering - FIFO) читання подій для конкретного замовлення одним і тим самим консьюмером.
6. **Saga Deadlines (Таймаути транзакцій):** Якщо зовнішня система (наприклад, платіжний шлюз Stripe) "зависає" і не повертає жодної відповіді, транзакція не може чекати вічно. Оркестратор має вбудований механізм таймаутів (зазвичай 14 хвилин — трохи менше за TTL семантичного блокування). Якщо таймер спрацьовує, Оркестратор вважає транзакцію неуспішною і проактивно розсилає команди на скасування (`[Cancel_Order_Cmd]`), перериваючи "зомбі-замовлення" та інформуючи клієнта у **мобільному застосунку** про скасування через тайм-аут.

---

### 4.4. Динамічна маршрутизація зовнішніх провайдерів (Smart Routing: Payments & Logistics)

Оскільки **2BeMarket** функціонує як глобальна екосистема, інтеграція з єдиним провайдером платежів або доставки є антипатерном. Замість цього `Order Service` взаємодіє з абстрактними шлюзами (**Payment Router** та **Logistics Gateway**), які реалізують динамічну маршрутизацію (Smart Routing):

- **Локалізація та Глобальні гравці:** Маршрутизатор аналізує `region` та валюту замовлення. Трафік з Америки та Європи направляється через глобальні шлюзи (Stripe, PayPal, Apple Pay / Google Pay).
- **Специфіка Азії (APAC):** Для користувачів з материкового Китаю та Південно-Східної Азії система автоматично перемикається на локальні інтеграції — **Alipay** або **WeChat Pay**, що розгорнуті в межах інфраструктури Alibaba Cloud.
- **Специфіка Близького Сходу (MENA):** Трафік з ОАЕ або Катару маршрутизується через сертифіковані локальні шлюзи, які підтримують принципи ісламського банкінгу (Sharia-compliant).
- **Автоматичний Failover:** Якщо основний еквайринг для регіону тимчасово недоступний (наприклад, Stripe повертає 503), Payment Router автоматично та непомітно для **мобільного застосунку** перекидає транзакцію на резервний шлюз (наприклад, Braintree), максимізуючи конверсію успішних оплат.
- **Logistics Gateway (Агрегація доставки):** Аналогічно до платежів, платформа не має жорсткої прив'язки до однієї служби доставки. `Order Service` взаємодіє з абстрактним **Logistics Gateway**. Під час оформлення замовлення цей шлюз динамічно аналізує габарити, вагу, адресу доставки та поточне завантаження провайдерів (DHL, FedEx, UPS або локальні кур'єрські служби), автоматично обираючи оптимального перевізника на основі вартості та заявленого SLA (Service Level Agreement).

---

### 4.5. "Точка неповернення" (Pivot Transaction) та Forward Recovery

Більшість класичних реалізацій Saga покладаються виключно на відкат назад (Backward Recovery). Проте в архітектурі **2BeMarket** впроваджено концепцію **Pivot Transaction (Точка неповернення)**, якою є успішне списання коштів платіжним шлюзом.

- **До Pivot-транзакції (Backward Recovery):** Якщо помилка стається на етапі резервації товару, система робить класичний Rollback — скасовує замовлення і звільняє ресурси.
- **Після Pivot-транзакції (Forward Recovery):** Якщо клієнт успішно оплатив товар, але фінальна подія `[Commit_Inventory]` (остаточне списання зі складу) не вдалася через падіння бази `Inventory Service`, система **НЕ робить автоматичний Refund** (оскільки це несе бізнес-втрати та комісії еквайрингу). Замість цього застосовується **Forward Recovery**:
    1. Повідомлення переміщується у **Dead Letter Queue (DLQ)**.
    2. Система робить експоненційні спроби (Exponential Backoff) повторити операцію.
    3. Якщо автоматика не справляється, сповіщається сервіс ручної звірки (Reconciliation Worker). Оператор складу або служба підтримки вирішує проблему вручну (наприклад, пропонує клієнту альтернативний товар).
- **Фінальна компенсуюча транзакція (Fall-back Compensation):** Незважаючи на те, що система надає пріоритет механізмам Forward Recovery (наприклад, пропозиція альтернативного товару або ручна звірка для покращення клієнтського досвіду), архітектура **2BeMarket** жорстко регламентує наявність Refund-у, як фінального компенсуючого кроку (Fall-back Compensation). У разі відмови клієнта від альтернатив, фізичного знищення товару або з метою дотримання законодавства щодо захисту прав споживачів, Saga-оркестратор ініціює компенсуючі транзакції для гарантованого та автоматизованого повернення коштів.

*Клієнт у цей час бачить статус "Оплачено, готується до відправки", що зберігає бездоганний User Experience, поки система вирішує консистентність "під капотом".*

---

### 4.6. Стихійні лиха та проблема розколу мережі (Split-Brain)

Глобальна інфраструктура вразлива до фізичного пошкодження магістральних каналів зв'язку (наприклад, розриву трансокеанічних оптоволоконних кабелів через землетруси чи геополітичні фактори). Якщо дата-центр у Європі втрачає зв'язок зі США, виникає критичний ризик **Split-Brain**: обидва регіони можуть вважати себе активними (Primary) і незалежно приймати платежі за той самий товар, що призведе до незворотного пошкодження бази даних.

Щоб уникнути цього, транзакційне ядро **2BeMarket** покладається на **Quorum Consensus** (алгоритм Raft). Однак, щоб подолати обмеження швидкості світла і не додавати 200+ мілісекунд трансатлантичної затримки до кожної покупки, використовується **Topology-Aware Geo-Partitioning**. 
Транзакції звичайних користувачів ніколи не чекають глобального підтвердження від інших континентів. Дані європейських клієнтів фізично "прив'язані" до європейського кластера, і кворум збирається виключно між трьома ізольованими зонами доступності (Availability Zones) *всередині* Європи (наприклад, Frankfurt-1, Frankfurt-2, Frankfurt-3). Трансатлантична реплікація в США чи Азію відбувається асинхронно у фоновому режимі (для Disaster Recovery). Глобальний кворум (Global Paxos) застосовується лише для критичних налаштувань платформи (наприклад, зміни глобальних податкових ставок), де консистентність важливіша за мілісекунди. Це гарантує 100% захист від Split-Brain без шкоди для продуктивності **мобільного застосунку**.

---

### 4.7. Захист БД від проблеми "Гарячого рядка" (Hot Row / Thundering Herd)

Специфіка електронної комерції (особливо флеш-розпродажі та Hype Drops) створює аномальні навантаження: десятки тисяч користувачів можуть одночасно намагатися зарезервувати один і той самий популярний товар. На рівні транзакційної бази даних це призводить до проблеми "Hot Row" — тисячі потоків блокують один одного, очікуючи доступу до одного рядка таблиці `INVENTORY`, що призводить до вичерпання пулу з'єднань та каскадного падіння (Cascading Failure).

Для вирішення цієї проблеми `Inventory Service` використовує комбінований підхід:

1. **Попереднє відсікання в пам'яті (Redis INCR/DECR):** Перш ніж транзакція Saga взагалі торкнеться реляційної бази даних, доступність залишку атомарно перевіряється в Redis. Якщо залишок у Redis дорівнює нулю, запит відхиляється за <2 мс.
2. **SKIP LOCKED у транзакційній БД:** Для фінального семантичного блокування в CockroachDB використовується SQL-конструкція `SELECT ... FOR UPDATE SKIP LOCKED`. Якщо 1000 запитів прорвалися до бази одночасно, лише перший захопить блокування рядка. Інші 999 запитів не будуть ставати в мертву чергу (очікуючи зняття блокування), а миттєво отримають помилку і будуть перенаправлені Оркестратором у статус `OUT_OF_STOCK`, повертаючи клієнтам у **мобільний застосунок** швидку і чітку відповідь замість нескінченного зависання екрана завантаження.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **5. Масштабування, Кешування та Управління Піковими Навантаженнями**

Проект **2BeMarket** спроектований для витримування пікових навантажень до **175,000 RPS**. Проте характер трафіку в електронній комерції є нерівномірним, тому архітектура використовує адаптивні стратегії масштабування та багаторівневе кешування.

### 5.1. Стратегії масштабування та Оптимізація витрат (FinOps)

Для ефективного управління ресурсами кластера та хмарними бюджетами, архітектура відмовляється від повільного стандартного масштабування по CPU. Замість цього використовується **KEDA (Kubernetes Event-driven Autoscaling)**. Це дозволяє миттєво масштабувати мікросервіси на основі реальних бізнес-метрик (наприклад, довжини черги запитів у Cloudflare або кількості нерозділених повідомлень у Kafka), випереджаючи навантаження ще до того, як процесори серверів почнуть деградувати.

Нижче наведено візуалізацію концепції **Asymmetric Scaling** (Асиметричного масштабування), яка демонструє різницю між глобальним сплеском трафіку та локальним регіональним розпродажем.

```mermaid
graph TD
    subgraph Legend ["Ключ до схеми"]
        H_P["🔥 Пікова потужність кластера<br>(Максимальні витрати)"]
        B_P["💤 Базова потужність кластера<br>(Мінімальні витрати)"]
    end

    subgraph Global_Sale ["СЦЕНАРІЙ 1: Global Black Friday (Всі регіони одночасно)"]
        Spacer1[" "]
        AWS_US_G[("🇺🇸 AWS USA<br>Кластер Order Service")]
        GCP_EU_G[("🇪🇺 GCP EU<br>Кластер Order Service")]
        AZURE_MEA_G[("🌍 Azure MENA<br>Кластер Order Service")]
        ALI_ASIA_G[("🇨🇳 Alibaba Asia<br>Кластер Order Service")]
    end

    subgraph Regional_Sale ["СЦЕНАРІЙ 2: Singles' Day (Розпродаж 11.11 тільки в Азії)"]
        Spacer2[" "]
        AWS_US_R[("🇺🇸 AWS USA<br>Кластер Order Service")]
        GCP_EU_R[("🇪🇺 GCP EU<br>Кластер Order Service")]
        AZURE_MEA_R[("🌍 Azure MENA<br>Кластер Order Service")]
        ALI_ASIA_R[("🇨🇳 Alibaba Asia<br>Кластер Order Service")]
    end

    %% Styling for Scenario 1
    class AWS_US_G,GCP_EU_G,AZURE_MEA_G,ALI_ASIA_G peak;

    %% Styling for Scenario 2 - THIS IS ASYMMETRIC
    class ALI_ASIA_R peak;
    class AWS_US_R,GCP_EU_R,AZURE_MEA_R base;

    %% Робимо розпірку повністю прозорою
    classDef invisible fill:none,stroke:none;
    class Spacer1 invisible;
    class Spacer2 invisible;

    %% Styling Definitions
    classDef peak fill:#3a0000,stroke:#ff0000,stroke-width:3px,color:#fff,rx:5;
    classDef base fill:#1a1a1a,stroke:#888,stroke-width:1px,color:#fff,rx:5;
    classDef legend_peak fill:#3a0000,stroke:#ff0000,stroke-width:3px,color:#fff;
    classDef legend_base fill:#1a1a1a,stroke:#888,stroke-width:1px,color:#fff;

    class H_P legend_peak;
    class B_P legend_base;
```

> 🌍 **Архітектурна примітка (Global Edge Routing):** На діаграмі зображені лише Core-регіони транзакційного ядра. Користувачі з інших континентів (Австралія, Африка, Південна Америка) обслуговуються через локальні Edge-вузли (Cloudflare PoPs). Статичний та кешований трафік віддається їм локально (затримка \<20мс), а транзакційні POST-запити маршрутизуються до найближчого Core-регіону через виділені оптичні магістралі (Cloudflare Argo Smart Routing), що забезпечує глобальне покриття без необхідності розгортати дорогі транзакційні БД на кожному континенті.

Управління ресурсами Kubernetes (HPA - Horizontal Pod Autoscaler) та базами даних налаштовано відповідно до бізнес-календаря:

1. **Глобальні розпродажі (Black Friday / Cyber Monday):**
    - *Сценарій:* Синхронний сплеск трафіку по всьому світу.
    - *Дія:* Глобальний **Pre-warming** (попереднє прогрівання). За 24 години до старту система штучно масштабує вузли у всіх 4-х хмарах (AWS, GCP, Azure, Alibaba) та завантажує найпопулярніші товари в Redis, щоб уникнути ефекту "холодного старту" (Cache Stampede).
2. **Регіональні розпродажі (Асиметричне масштабування):**
    - *Сценарій:* Локальні свята (наприклад, Singles' Day 11.11 в Азії, Рамадан на Близькому Сході, День Подяки у США).
    - *Дія:* Для оптимізації витрат (FinOps) система застосовує **Asymmetric Scaling**. Якщо відбувається розпродаж 11.11, трафік і ресурси масштабуються виключно в кластерах Alibaba Cloud (Азія), тоді як GCP (Європа) та інші хмари залишаються на базовому рівні забезпечення. *Це ключова бізнес-перевага нашої архітектури, показана на діаграмі вище.*

---

### 5.2. Контрольована деградація (Graceful Degradation)

Якщо під час Black Friday навантаження перевищить проектні 175,000 RPS, система автоматично переходить у режим самозбереження (Graceful Degradation), щоб врятувати транзакційне ядро:

- **Вимкнення "важких" фіч:** Алгоритми ML-рекомендацій ("З цим товаром купують") тимчасово вимикаються для користувачів **мобільного застосунку**. На їх місце підставляється статичний кешований список бестселерів.
- **Уповільнення некритичних фонових задач:** Генерація аналітичних звітів для продавців призупиняється до спаду навантаження, звільняючи CPU для мікросервісу `Order Service`.

---

### 5.3. Багаторівнева архітектура захисту від пікових навантажень

Нижче наведено технічну діаграму проходження запиту від Клієнта до Транзакційної Бази Даних під час запуску ексклюзивних товарів (Hype Drop). Діаграма ілюструє, як працює наша багаторівнева система захисту, відсікаючи навантаження на кожному етапі.

```mermaid
sequenceDiagram
    autonumber
    actor Client as 📱 Клієнт (Mobile/Web)
    participant Edge as 🛡️ Рівень Edge (Cloudflare)
    participant Worker as ⚙️ Cloudflare Worker<br>(KV Store)
    participant Gateway as 🌐 API Gateway
    participant Redis as ⚡ Redis Cluster (Idempotency)
    participant OrderService as 📦 Order Service<br>(Orchestrator)
    participant DB as 🗄️ CockroachDB<br>(Outbox)
    participant Kafka as ⚡ Apache Kafka
    participant PushGateway as 📡 Push Gateway

    Note over Client, Edge: --- СЦЕНАРІЙ: Hype Drop (Секунда релізу) ---

    Client->>Edge: GET /product/iphone-17 (Мільйон запитів)
    Edge->>Worker: Перевірка Feature Flag (is_released?)

    alt Ембарго ще діє (is_released = false)
        Worker-->>Edge: 404 Not Found (Маскування кешу)
        Edge-->>Client: 404 Not Found (За <10мс)
        Note right of Edge: 🔥 Бекенд відпочиває (0 RPS)
    else Товар релізнуто (The Flip)
        Worker-->>Edge: True
        Edge-->>Client: 200 OK (Контент із CDN Кешу)
        Note right of Edge: ✅ Мільйон користувачів бачать товар<br>(GET не долітають до БД)
    end

    Note over Client, PushGateway: --- Користувачі натискають 'Купити' (Write Surge) ---

    Client->>Edge: POST /checkout (Write Surge - некешовані)

    Note over Edge: 🧱 ПАТЕРН: Virtual Waiting Room<br>(Edge-Level Queuing)
    Edge-->>Client: Екран 'Ви в черзі' (Крипто-токен)<br>Бекенд захищено від шторму

    Note over Edge, PushGateway: --- Клієнт проходить чергу (Rate Limited) ---

    Edge->>Gateway: POST /checkout (Токен OK, 1000 RPS)
    Gateway->>Redis: Перевірка ідемпотентності
    Redis-->>Gateway: Cache Miss

    Note over Gateway, DB: 📦 Асинхронна Saga (Див. Розділ 4)
    Gateway->>OrderService: Ініціалізація Saga (PENDING)
    OrderService-->>Gateway: 202 Accepted (Order ID)
    Gateway-->>Client: HTTP 202: "Обробляємо..." (UI: Loader)

    Note over OrderService, PushGateway: --- Фонова обробка (Event-Driven) ---
    OrderService->>DB: Запис PENDING (Transactional Outbox)
    DB-->>OrderService: OK
    OrderService->>Kafka: Publish [Start_Saga_Events]
    Note right of Kafka: Оплата та Склад обробляються<br>асинхронно (див. Розділ 4)
    Kafka->>PushGateway: [Order_Paid] (Від Payment Router)
    PushGateway-->>Client: WebSocket: 'Замовлення успішне! ✅'
```

> 🧱 **Архітектурна примітка (Connection Multiplexing):** Щоб 1,000 безпечних RPS, які пройшли через віртуальну чергу, не знищили транзакційну базу даних через вичерпання TCP-з'єднань (Connection Churn), мікросервіси ніколи не підключаються до БД напряму. Перед транзакційним ядром розгорнуто пул з'єднань (наприклад, **PgBouncer** або вбудований проксі CockroachDB). Він підтримує фіксовану кількість постійних з'єднань із базою (наприклад, 200) і мультиплексує через них тисячі клієнтських транзакцій, захищаючи оперативну пам'ять та процесор БД від накладних витрат на відкриття/закриття сокетів.

---

### 5.4. Багаторівневе кешування (Multi-Tier Caching)

На основі архітектури, показаної на діаграмі в пункті 5.3, щоб мінімізувати кількість звернень до баз даних і забезпечити затримку (latency) \< 50мс навіть під час пікових навантажень, застосовується трирівнева стратегія кешування:

1. **Edge Caching (Cloudflare CDN):** Статичний контент (зображення товарів, відео-огляди, CSS/JS) та публічні API-відповіді (наприклад, сторінка "Топ-10 товарів дня") кешуються на рівні CDN. Це відсікає до 60% загального трафіку ще до того, як він досягне наших хмарних серверів. *(Для релізу ексклюзивних товарів цей рівень доповнюється патерном Embargo Caching, див. п. 5.5)*
2. **Distributed Caching (Redis Cluster):** Використовується патерн *Cache-Aside* для динамічних, але рідко змінюваних даних (наприклад, ціни та описи товарів з ElasticSearch). Якщо ціна оновлюється, сервіс інвалідує ключ у Redis.
3. **Local In-Memory Caching:** Для супер-гарячих даних, які потрібні для кожної транзакції (наприклад, таблиці податкових ставок або конфігурації платформи), використовується локальний кеш всередині пам'яті самого мікросервісу (наприклад, Caffeine Cache).

---

### 5.5. Патерн Embargo Caching (Секретне прогрівання кешу)

Під час запуску ексклюзивних товарів (наприклад, лімітованих колекцій кросівок або нових гаджетів) виникає архітектурний парадокс: контент потрібно закешувати на глобальних CDN-вузлах до старту розпродажу, щоб уникнути падіння бекенду, але він не повинен бути доступним для публіки (навіть якщо хтось вгадає URL-адресу).

Для вирішення цього **2BeMarket** використовує патерн **Embargo Caching** на базі Edge Computing (Cloudflare Workers + Workers KV):

1. **Зашифроване прогрівання:** За 24 години до релізу статичні асети та JSON-відповіді з цінами завантажуються в кеш CDN.
2. **Edge Authorization:** Усі запити до цих ресурсів перехоплюються скриптом Cloudflare Worker, який перевіряє глобальний прапорець (Feature Flag) у надшвидкому сховищі Workers KV (наприклад, `is_product_x_released = false`). Поки прапорець опущений, Worker віддає `404 Not Found`, маскуючи наявність кешу.
3. **The Flip (Миттєвий реліз):** У секунду старту розпродажу оркестратор змінює прапорець на `true`. Зміна поширюється глобальною мережею Edge-вузлів за \<10 мс. Усі наступні запити миттєво отримують реальний контент прямо з кешу CDN. Це гарантує, що в момент релізу мільйони GET-запитів не долетять до наших серверів.
    - *Результат:* Мільйони користувачів бачать новий товар одночасно, а транзакційне ядро (AWS/GCP) не отримує жодного зайвого запиту на читання під час "відкриття вітрини".

---

### 5.6. Управління хайп-трафіком (Edge-Level Waiting Room)

Патерн Embargo Caching ідеально захищає інфраструктуру від перевантаження запитами на читання (GET). Однак у момент релізу мільйони користувачів одночасно натискають кнопку "Купити", генеруючи лавину некешованих POST-запитів, здатних знищити транзакційну базу даних (Order Service).

Для захисту від шторму запитів на запис **2BeMarket** використовує патерн **Virtual Waiting Room** (Віртуальна черга на рівні CDN):

1. **Edge-Queuing:** У момент пікового навантаження Cloudflare перехоплює трафік до ендпоінту `/checkout` і автоматично формує криптографічну живу чергу на глобальних Edge-вузлах, не пропускаючи трафік до бекенду.
2. **Backpressure (Зворотний тиск):** Наш оркестратор Kubernetes моніторить поточне навантаження на БД (наприклад, тримає його на рівні 80% від максимуму). Залежно від цього, CDN видає "перепустки" (Tokens) у транзакційне ядро строго дозованими партіями (наприклад, рівно 1000 покупців на секунду).
3. **Smart UX:** Користувачі в **мобільному застосунку** не бачать помилку "503 Service Unavailable". Замість цього вони бачать екран "Ви в черзі за ексклюзивним товаром. Ваш орієнтовний час очікування: 2 хвилини". Це зберігає лояльність аудиторії та гарантує 100% успішність транзакцій для тих, хто пройшов чергу.

**Real-time комунікація та вирішення проблеми C10M (Push Gateway):** 
Для відображення актуального статусу в "живій черзі" або оновлення залишків на складах під час Hype Drops, система використовує асинхронну push-модель на базі WebSockets / Server-Sent Events (SSE). 

Проте, архітектурним антипатерном є пряме підключення мільйонів користувачів до бізнес-мікросервісів (наприклад, `Order Service`). Це призвело б до вичерпання пулу сокетів та оперативної пам'яті (класична проблема C10M). Для вирішення цього завдання **2BeMarket** використовує виділений кластер **Push Gateway** (наприклад, Centrifugo або керований AWS API Gateway WebSockets). 

Цей кластер написаний на Go/Rust, оптимізований виключно для утримання мільйонів "легких", бездіяльних TCP-з'єднань. Він працює як Pub/Sub: підписується на події з глобальної шини Apache Kafka і транслює їх у **мобільний застосунок** клієнта в реальному часі. Бізнес-мікросервіси залишаються повністю Stateless і захищеними від мережевого перевантаження.

---

### 5.7. Рівень даних: Inventory Sharding для екстремальних Hype Drops

У Розділі 4.7 ми впровадили механізм `SKIP LOCKED` для усунення дедлоків під час конкурентного доступу до таблиці `INVENTORY_LEDGER`. Проте, під час екстремальних Hype Drops (наприклад, реліз нової консолі), коли система пропускає 1,000 транзакцій на секунду на *один і той самий товар*, навіть `SKIP LOCKED` може призвести до надто високого відсотка відхилених запитів (Transaction Retries), оскільки фізична межа запису в один рядок бази досягається миттєво.

**Рішення (Inventory Sharding / Розподілені лічильники):** Щоб зняти фізичне обмеження дискового I/O та усунути вузьке місце, **2BeMarket** доповнює логіку `SKIP LOCKED` патерном декомпозиції залишків (Row Sharding). Замість того, щоб зберігати 10,000 одиниць ексклюзивного товару в одному рядку, система автоматично "розмазує" цей залишок на 100 окремих віртуальних рядків (бакетів) по 100 одиниць у кожному (наприклад, `product_id_bucket_1`, `product_id_bucket_2` і т.д.).

Коли користувач через **мобільний застосунок** ініціює покупку, транзакція `Order Service` випадковим чином (через хеш-функцію) обирає один із 100 бакетів для списання. Це геніальне та просте рішення розпаралелює навантаження на транзакційну базу даних у 100 разів, повністю усуваючи "гарячі точки" (Hotspots) і дозволяючи системі миттєво фіксувати тисячі паралельних продажів одного товару без жодного Deadlock-у.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **6. Безпека, Комплаєнс та Захист API (Security & Data Privacy)**

Для глобальної платформи електронної комерції безпека поділяється на три вектори: юридичний комплаєнс (захист персональних даних та фінансів), захист інфраструктури від цілеспрямованих атак та безпека веб- та мобільних платформ. Архітектура **2BeMarket** реалізує концепцію *Security by Design* (Безпека на рівні проектування).

### 6.1. Глобальний Комплаєнс (GDPR, nFADP та PCI-DSS)

Оскільки платформа оперує на ринках Європи, вона повинна суворо дотримуватись законів про захист даних.

1. **Data Residency (Регіональне зберігання):** Дані європейських користувачів фізично не залишають дата-центри в ЄС (GCP Frankfurt). Це забезпечує відповідність вимогам **GDPR** (Євросоюз) та суворому швейцарському закону **nFADP** (New Federal Act on Data Protection).
2. **PII Tokenization (Токенізація персональних даних):** У транзакційних базах (Order Service) не зберігаються імена, імейли чи номери телефонів у відкритому вигляді. Використовується криптографічна токенізація. Якщо хакер отримає дамп таблиці `ORDERS`, він побачить лише ідентифікатори (UUID), позбавлені контексту.
3. **PCI-DSS Scope Reduction:** Система ніколи не торкається "сирих" номерів кредитних карток (PAN). **Мобільний застосунок** та Web-фронтенд відправляють дані картки безпосередньо у платіжний шлюз (наприклад, Stripe), отримуючи натомість безпечний токен для подальших списань.
4. **Право на забуття (Патерн Crypto-Shredding):** Найскладніша вимога GDPR та nFADP — це видалення даних користувача на його запит, оскільки дані можуть знаходитись у сотнях холодних бекапів. **2BeMarket** вирішує це математично: PII кожного клієнта шифрується його індивідуальним ключем (KMS). При запиті на видалення система просто знищує цей ключ. Усі історичні записи, бекапи та логи миттєво перетворюються на нечитабельний криптографічний шум, забезпечуючи 100% compliance без сканування та переписування петабайтів даних.

---

### 6.2. Захист від ботів та DDoS (Scalper Bot Protection)

Під час запуску ексклюзивних товарів (наприклад, лімітованих кросівок або нових смартфонів) до 80% трафіку можуть складати боти-скальпери (Scalper Bots), які намагаються викупити весь сток за мілісекунди.

- **Cloudflare Bot Management:** Перед API Gateway стоїть інтелектуальний WAF (Web Application Firewall), який аналізує поведінку користувача (рух миші, швидкість набору, TLS-відбитки). Підозрілий трафік отримує невидимий JS-виклик (Challenge) або CAPTCHA.
- **Adaptive Rate Limiting:** Обмеження кількості запитів реалізовано не лише за IP-адресою, а й за `user_id` та токеном сесії (щоб запобігти розподіленим атакам через ботнети). Наприклад, ендпоінт `POST /checkout` дозволяє не більше 2 запитів на секунду від одного користувача.
- **GraphQL Query Depth & Complexity Analysis:** Оскільки платформа використовує GraphQL для агрегації каталогу, існує ризик атак через рекурсивні або глибоко вкладені запити (наприклад, запит товарів, їхніх відгуків, авторів відгуків, інших товарів цих авторів і т.д.), що може "покласти" базу даних. Для захисту від цього API Gateway аналізує абстрактне синтаксичне дерево (AST) кожного GraphQL-запиту ще до його виконання. Запити з глибиною вкладеності (Query Depth) більше 5 або з високою оцінкою обчислювальної складності (Query Complexity Score) автоматично відхиляються з помилкою 400 Bad Request.

---

### 6.3. Концепція Zero Trust та Багаторівневий захист Клієнтів, API і Мікросервісів

Захист комунікацій реалізовано на двох фундаментальних рівнях: зовнішньому (Edge) та внутрішньому (Cluster), з урахуванням специфіки кожної клієнтської платформи:

Нижче наведено діаграму, яка ілюструє, як архітектура **2BeMarket** реалізує концепцію "Нульової довіри" (Zero Trust), поєднуючи захист на рівні Edge (Cloudflare) та Cluster (Kubernetes).

```mermaid
graph LR
    subgraph External_Network ["Зовнішня Мережа"]
        Client(("📱 Клієнт<br>(Mobile/Web)"))
    end

    subgraph Edge ["🛡️ Рівень Edge (Cloudflare)"]
        WAF{{"🔍 Bot Management<br>& WAF Перевірка"}}
    end
    
    subgraph Cluster ["☸️&nbsp;Рівень&nbsp;Cluster&nbsp;(Kubernetes)"]
        Gateway["🌐 API Gateway<br>(JWT Валідація RS256)"]
        OrderService["📦 Order Service"]
        InventoryService["🛒 Inventory Service"]
    end

    %% Flows
    Client -.->|🔒 HTTPS + Pinnig| WAF
    WAF -.->|✅ Трафік OK| Gateway

    %% Internal Flows - Service Mesh
    Gateway ==>|"🔑 mTLS (Istio)"| OrderService
    OrderService ==>|"🔑 mTLS (Istio)"| InventoryService

    %% Styling
    classDef external fill:#2b2b2b,stroke:#ccc,color:#fff,rx:5px
    classDef edge fill:#3a0000,stroke:#ff0000,stroke-width:2px,color:#fff
    classDef cluster fill:#0d47a1,stroke:#64b5f6,stroke-width:2px,color:#fff
    classDef internal fill:#1a1a1a,stroke:#ccc,color:#fff

    class Client external
    class WAF edge
    class Gateway cluster
    class OrderService internal
    class InventoryService internal

    linkStyle 2,3 stroke-width:4px,fill:none,stroke:gold;
```

**Опис багаторівневих механізмів "Нульової довіри" (Zero Trust):**

1. **Захист клієнтів (Certificate Pinning & CSP):** На рівні Edge мобільний трафік захищений від перехоплення (Man-in-the-Middle) за допомогою вшитого в **мобільний застосунок** хешу SSL-сертифіката (Pinning). Для Web-порталу діють суворі заголовки безпеки (HSTS, CSP), що унеможливлюють виконання шкідливих скриптів (XSS), а сесійні токени передаються виключно через `Secure HttpOnly` cookies.
2. **Автентифікація (OAuth 2.0 + JWT):** API Gateway виступає як єдина точка входу і перевіряє криптографічний підпис JWT-токена (RS256). Внутрішні мікросервіси довіряють Gateway і не витрачають CPU на повторну валідацію токенів.
3. **Zero Trust Architecture (mTLS):** Усередині Kubernetes-кластера немає "довіреної зони" (показано товстими золотими стрілками). Комунікація між мікросервісами жорстко зашифрована за допомогою mTLS (Mutual TLS), де кожен сервіс автентифікує іншого через Service Mesh (Istio / Linkerd).
4. **Захист від BOLA / IDOR (Авторизація на рівні об'єкта):** Наявність валідного токена не дає права читати чужі дані. Для захисту від найнебезпечнішої API-вразливості (Broken Object Level Authorization), мікросервіси використовують патерн ABAC. При запиті `GET /orders/{order_id}`, сервіс криптографічно звіряє `user_id` з токена з полем `owner_id` цільового об'єкта.

---

### 6.4. Захист ланцюга постачання та Динамічні секрети (Shift-Left Security)

Справжня безпека рівня FAANG виходить з парадигми "Assume Breach" (припустимо, що нас уже зламали). Тому архітектура **2BeMarket** жорстко обмежує радіус ураження (Blast Radius) у разі компрометації окремого контейнера чи розробника:

1. **Dynamic Secrets (Жодних статичних паролів):** У системі фізично не існують константи з паролями до баз даних. Замість цього використовується **HashiCorp Vault**. Коли `Order Service` хоче записати дані, він запитує у Vault тимчасові (Just-In-Time) облікові дані, які автоматично знищуються через 15 хвилин. Навіть якщо зловмисник зробить дамп пам'яті контейнера, вкрадені доступи стануть марними майже миттєво.
2. **Workload Identity (IRSA / Workload Federation):** Мікросервіси не використовують статичні AWS Access Keys чи GCP Service Accounts. Замість цього застосовується прив'язка Kubernetes Service Accounts до хмарних IAM-ролей. Кожен Pod отримує тимчасовий STS-токен виключно для тих ресурсів (наприклад, конкретного S3 бакета), які йому потрібні (Principle of Least Privilege).
3. **Supply Chain Security (Захист від ін'єкцій у код):** Щоб унеможливити атаки типу SolarWinds, процес CI/CD повністю ізольований:
    - Усі мікросервіси пакуються у **Distroless** контейнери (в них фізично відсутній `bash` чи `sh`, що унеможливлює віддалене виконання команд — RCE).
    - Кожен Docker-образ криптографічно підписується в пайплайні за допомогою **Sigstore/Cosign**.
    - Кластер Kubernetes має вбудований *Admission Controller*, який на рівні ядра забороняє розгортання будь-якого контейнера, якщо він не має валідного цифрового підпису від нашого офіційного CI-сервера.

---

### 6.5. ML Risk Engine (Захист від Фроду та Account Takeover)

Традиційний WAF безсилий проти логічного фроду — використання вкрадених кредитних карток (Carding) або зламів акаунтів (Account Takeover - ATO). Для захисту бізнес-метрик, **2BeMarket** інтегрує асинхронний **ML Risk Engine**:

- **Динамічний Risk Scoring:** Кожна дія в **мобільному застосунку** або на вебі аналізується ML-моделлю в реальному часі (< 20 мс), враховуючи сотні сигналів (швидкість набору, рухи миші, розбіжності між Geo-IP і BIN-кодом картки).
- **Adaptive Friction:** Якщо Risk Score низький, транзакція проходить миттєво. Якщо система помічає аномалію, Risk Engine тригерить додаткову перевірку — вимагає біометричне підтвердження в **мобільному застосунку** або ініціює жорсткий виклик 3D Secure.

---

### 6.6. Глобальні санкції та Anti-Money Laundering (AML / KYC)

Оскільки **2BeMarket** оперує транскордонними фінансовими потоками, платформа підпадає під жорстке регулювання міжнародних інституцій. Архітектура включає автоматизований контур фінансового комплаєнсу:

- **Geo-Fencing на рівні Edge:** Трафік з IP-адрес підсанкційних країн відсікається ще на рівні балансувальників Cloudflare (HTTP 403), не доходячи до мікросервісів.
- **Скринінг продавців (KYB/KYC):** Сервіс `Onboarding Worker` асинхронно перевіряє юридичних осіб та їх бенефіціарів (UBO) через глобальні бази даних ризиків (LexisNexis).
- **Real-Time Sanctions Screening:** Кожна транзакція проходить перевірку через санкційні списки OFAC (США) та ЄС. При збігу транзакція миттєво заморожується (`FROZEN_COMPLIANCE`) до ручного розгляду комплаєнс-офіцером.

---

### 6.7. Незмінні Аудит-Логи (Immutable Audit Trail) та SIEM

Виходячи з парадигми "Assume Breach", архітектура враховує сценарій компрометації найвищих привілеїв (наприклад, дії інсайдера або крадіжка root-доступу). Для забезпечення чистоти розслідувань (Digital Forensics):

- **Centralized Security Logging:** Усі події авторизації, зміни прав доступу (IAM) та доступи до PII-даних асинхронно надсилаються до ізольованої системи SIEM (Security Information and Event Management).
- **WORM Storage (Write-Once-Read-Many):** На відміну від бекапів бази даних (які захищають *стан* системи), сирі аудит-логи безпеки архівуються у спеціальні "заморожені" бакети Amazon S3 з увімкненим режимом *Object Lock (Compliance Mode)*. Цей апаратний механізм гарантує, що жодна людина у світі — навіть CEO чи інженер з root-правами — не зможе змінити або видалити файл логу безпеки до закінчення законодавчо встановленого терміну зберігання.

---

### 6.8. Trust & Safety (Автоматична модерація контенту)

Для глобального маркетплейсу, де тисячі продавців щодня завантажують нові товари (UGC - User Generated Content), критичним вектором ризику є публікація забороненого контенту (18+, зброя, наркотичні речовини, контрафакт або ненормативна лексика). Наявність таких товарів призводить до негайного блокування платіжних шлюзів (Stripe/Visa) та видалення **мобільного застосунку** з Apple App Store та Google Play.

Щоб унеможливити потрапляння такого контенту на вітрину, архітектура **2BeMarket** впроваджує асинхронний пайплайн модерації (Content Moderation Pipeline) на базі штучного інтелекту:

1. **Pre-Publication Quarantine (Карантин):** Коли продавець створює картку товару, вона не потрапляє в `Catalog Service` одразу. Замість цього товар отримує статус `PENDING_REVIEW` і відправляється в ізольовану чергу Kafka.
2. **Computer Vision & NLP Analysis:** Спеціалізований мікросервіс (`Moderation Worker`) паралельно проганяє всі медіафайли та тексти через ML-моделі (наприклад, AWS Rekognition або Google Cloud Vision API). 
    - Алгоритми комп'ютерного зору миттєво аналізують зображення на наявність Nudity, Violence або заборонених логотипів (захист від підробок).
    - NLP-моделі (Natural Language Processing) сканують опис та відгуки на наявність ненормативної лексики, hate speech або прихованих контактних даних (спроба продавця провести угоду поза платформою).
3. **Automated Enforcement & Human-in-the-Loop:**
    - Якщо модель впевнена на 99% у порушенні політики, товар автоматично видаляється (Hard Delete), а акаунт продавця отримує страйк.
    - Якщо товар проходить перевірку успішно, він автоматично публікується в каталозі.
    - Для "сірих" зон (Edge cases, впевненість моделі 50-80%) товар маршрутизується у внутрішню CRM для ручної перевірки командою модераторів (Trust & Safety Team). Це гарантує стерильність каталогу без уповільнення легального бізнесу.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **7. Обсервабільність, Моніторинг та SRE-практики (Site Reliability Engineering)**

Для системи, яка обробляє 175,000 RPS у гібридній мультихмарній екосистемі, класичного моніторингу (перевірки "чи живий сервер") недостатньо. Архітектура **2BeMarket** впроваджує концепцію **Deep Observability** (Глибокої обсервабільності), яка дозволяє миттєво знаходити першопричину (Root Cause) будь-якої аномалії серед тисяч ефемерних мікросервісів.

### 7.1. Розподілене трасування (Distributed Tracing)

Оскільки один клієнтський запит (наприклад, Checkout) ініціює ланцюг подій у десятках сервісів (Saga Pattern) та перетинає межі хмарних провайдерів (AWS, GCP), платформа використовує **OpenTelemetry** як єдиний стандарт збору телеметрії.

- **Trace ID Injection:** Коли **мобільний застосунок** ініціює запит, він генерує унікальний `x-trace-id` і передає його в заголовках HTTP. Цей ID "прошиває" весь шлях: від WAF на Cloudflare, через API Gateway, до топіків Kafka та фінального запису в CockroachDB.
- **Jaeger / Tempo Backend:** Усі спани (Spans) асинхронно відправляються до централізованого бекенду (наприклад, Grafana Tempo). Це дозволяє SRE-інженерам бачити візуальний водоспад (Waterfall) транзакції і миттєво розуміти, що затримка оплати виникла через 5-секундний таймаут у `Payment Router` при зверненні до зовнішнього API Stripe.

---

### 7.2. Метрики, SLI/SLO та Алертинг

Система моніторингу побудована на базі **Prometheus** (локальний збір) та **VictoriaMetrics** (глобальне довгострокове сховище), що дозволяє витримувати мільйони активних часових рядів (Time Series) без деградації продуктивності.

- **Golden Signals:** Моніторинг фокусується на чотирьох "Золотих сигналах" (Latency, Traffic, Errors, Saturation).
- **Service Level Objectives (SLO):** Замість абстрактних обіцянок, надійність системи вимірюється через жорсткі SLO. Наприклад: "99.9% запитів до `Catalog Service` повинні виконуватися швидше ніж за 100 мс". Якщо Error Budget (бюджет помилок) вичерпується, автоматика блокує розгортання нових фіч (Feature Freeze), поки команда не виправить проблеми з надійністю.
- **Smart Alerting (PagerDuty):** Щоб уникнути "втоми від алертів" (Alert Fatigue), система налаштована на симптоматичний алертинг. Інженер не отримує сповіщення "Завантаження CPU на Pod-і 95%" (це нормальна робота KEDA). Сповіщення (дзвінок посеред ночі) приходить лише тоді, коли "Відсоток успішних оплат впав нижче 98% за останні 3 хвилини".

---

### 7.3. Централізоване логування (Log Routing) з Data Masking

Усі операційні логи мікросервісів збираються в реальному часі агентами (FluentBit). Для дотримання вимог Розділу 6.7 щодо безпеки, агент виконує жорстку маршрутизацію (Log Routing) на рівні кластера:

- **Операційні логи (App & Error Logs):** Відправляються в центральний кластер (ELK Stack / Datadog) для щоденного дебагінгу SRE-командою.
- **Аудит-логи (Security Logs):** Ізолюються та маршрутизуються безпосередньо у SIEM-систему та незмінне WORM-сховище (Amazon S3).
- **PII Data Masking (Compliance):** Згідно з правилами захисту даних, на рівні агента (до відправки логів по мережі) налаштовані жорсткі RegEx-фільтри, які автоматично маскують номери кредитних карток, імейли та паролі (замінюючи їх на `[REDACTED]`).
- **Структуровані логи (JSON):** Мікросервіси ніколи не пишуть логи як простий текст. Усі логи формуються виключно у форматі JSON із фіксованою схемою (містять `timestamp`, `level`, `trace_id`, `service_name`), що дозволяє миттєво індексувати їх у глобальному сховищі та зв'язувати з трейсами OpenTelemetry.

---

### 7.4. GitOps та Безшовне розгортання (CI/CD Pipeline)

В умовах, коли система генерує прибуток щосекунди, зупинка на технічне обслуговування (Maintenance Window) є неприпустимою. Розгортання нових версій відбувається повністю автоматично і без простоїв (Zero Downtime).

1. **GitOps парадигма (ArgoCD):** Єдиним джерелом істини для інфраструктури є Git-репозиторій. Якщо розробник хоче змінити кількість реплік або оновити конфігурацію, він створює Pull Request. ArgoCD безперервно синхронізує стан кластера з Git-репозиторієм, автоматично відкочуючи будь-які ручні зміни, зроблені через консоль (захист від конфігураційного дрейфу).
2. **Canary Deployments (Канаркові релізи):** Нова версія мікросервісу (наприклад, оновлений алгоритм `Dynamic Pricing`) ніколи не викочується на 100% користувачів одразу.
    - Спочатку нова версія отримує 5% реального трафіку (наприклад, тільки користувачі з iOS).
    - Автоматика (наприклад, Flagger) протягом 15 хвилин порівнює метрики (відсоток HTTP 500, затримку) між старим і новим сервісами.
    - Якщо нова версія показує аномалії — трафік миттєво і непомітно для клієнтів перемикається назад на стабільну версію (Automated Rollback). Якщо все гаразд — відсоток поступово зростає до 100%.

---

### 7.5. Chaos Engineering (Інженерія Хаосу)

Найкращий спосіб переконатися, що система витримає катастрофу, — це штучно створювати катастрофи в робочий час, коли всі інженери онлайн і готові реагувати.

Платформа **2BeMarket** інтегрує практики Chaos Engineering (на базі інструментів типу Chaos Mesh / Gremlin):

- **Network Blackholing:** Автоматика періодично "обриває" зв'язок між `Order Service` та БД `Inventory` для 1% трафіку, щоб перевірити, чи коректно відпрацьовує механізм Exponential Backoff та Dead Letter Queue (DLQ), описаний у Розділі 4.5.
- **Pod Killing:** Випадкове знищення вузлів Kubernetes для тестування стійкості State Machine оркестратора Саги (перевірка того, що перервані транзакції успішно підхоплюються іншими вузлами).

> Цей проактивний підхід гарантує, що архітектурні патерни (Circuit Breaker, Failover, Timeout) реально працюють у Production, а не існують лише на діаграмах.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **8. Disaster Recovery (DRP), Business Continuity та FinOps**

Навіть найдосконаліша система з резервуванням може зіткнутися з подіями рівня "Чорний лебідь" (Black Swan events): відкликання ліцензій хмарного провайдера, глобальні атаки на BGP-маршрутизацію або стихійні лиха, що знищують цілі дата-центри. Тому архітектура **2BeMarket** регламентує жорсткі плани відновлення (Disaster Recovery Plan) та оптимізації витрат.

### 8.1. Цільові метрики відновлення (RTO та RPO)

Бізнес-логіка платформи поділена на рівні критичності (Tiers), для кожного з яких визначені суворі метрики: **RTO** (Recovery Time Objective — скільки часу система може лежати) та **RPO** (Recovery Point Objective — скільки даних ми можемо втратити).

- **Tier 1 (Транзакційне ядро, Баланси, Orders):**
    - **RPO = 0 (Zero Data Loss):** Втрата фінансових транзакцій є неприпустимою. Гарантується синхронною реплікацією CockroachDB через `Quorum Consensus` (Розділ 4.6).
    - **RTO < 5 хвилин:** Забезпечується автоматичним Failover-ом бази даних на вцілілу зону доступності (AZ) без втручання людини.
- **Tier 2 (Каталог товарів, Пошук, AI-рекомендації):**
    - **RPO < 15 хвилин:** Втрата останніх оновлень описів товарів не є критичною для фінансів.
    - **RTO < 30 хвилин:** Втрата кешу (Redis) або індексів (ElasticSearch) вимагає їх "перепрогрівання" (Rehydration) з основного сховища. До цього моменту застосунок працює в режимі Graceful Degradation (видача статичних бестселерів).
- **Tier 3 (Аналітика продавців, Відгуки, UGC):**
    - **RPO < 24 години:** Дані відновлюються зі щоденних незмінних WORM-бекапів на Amazon S3.
    - **RTO < 12 годин:** Час, необхідний на розгортання резервних кластерів та імпорт даних з холодного сховища.

---

### 8.2. Глобальний Failover та Стратегія Active-Active

Більшість класичних систем використовують стратегію Active-Passive (коли резервний дата-центр просто чекає на аварію, спалюючи гроші). **2BeMarket** використовує виключно топологію **Active-Active**.

- Усі макрорегіони (AWS USA, GCP EU, Alibaba Asia) постійно обробляють живий трафік.
- Якщо цілий європейський регіон (наприклад, GCP Frankfurt) повністю "зникає" з мережі, Anycast-маршрутизація Cloudflare (через health-чеки) за < 10 секунд виявляє відсутність маршруту.
- Європейський трафік автоматично перенаправляється до найближчого вцілілого кластера (наприклад, AWS USA East).
- Завдяки єдиному шару глобальної ідентифікації (Auth0 JWKS) та асинхронній реплікації станів по приватному бекбону (Розділ 2), європейські користувачі зберігають свої сесії та складені кошики (CRDT-вектори). **Проте, оскільки система суворо дотримується Data Sovereignty і не реплікує європейські персональні дані (PII) до США, для завершення транзакції під час глобального Failover-у застосунок переходить у режим "Guest Checkout", вимагаючи одноразового повторного введення адреси доставки.**

---

### 8.3. Економіка платформи та FinOps (Cloud Cost Management)

Підтримка інфраструктури, здатної витримати 175,000 RPS, може коштувати мільйони доларів щомісяця. Щоб платформа залишалася високомаржинальною, на рівні інфраструктурного коду (Terraform) закладено суворі політики FinOps:

1. **Spot-інстанси для Machine Learning та Аналітики:** Навчання моделей рекомендацій (ML Training), інференс для Feature Store та пакетна обробка аналітики (Apache Spark) не вимагають 100% гарантії безперервності обчислень. Для цих пулів Kubernetes-вузлів (Node Pools) використовуються виключно Spot/Preemptible інстанси, що знижує витрати на "важкі" ML-обчислення до **70-80%**. Якщо провайдер забирає Spot-інстанс, планувальник (Scheduler) просто перезапускає задачу на іншому доступному вузлі.
2. **ARM-архітектура (Graviton / Ampere):** Усі Stateless-мікросервіси (написані на Go/Rust) компілюються та розгортаються на процесорах архітектури ARM64 (наприклад, AWS Graviton3). Це дає в середньому на 20-30% краще співвідношення продуктивності до вартості (Price/Performance) порівняно з класичними x86-процесорами.
3. **Data Tiering (Життєвий цикл даних):** Зберігання петабайтів медіафайлів (фото/відео товарів) керується автоматичними політиками (S3 Lifecycle Rules). Якщо товар знімається з виробництва і не переглядається більше 90 днів, його медіа-асети автоматично переміщуються з гарячого сховища (S3 Standard) до найдешевшого архівного (S3 Glacier Deep Archive), що зменшує вартість зберігання гігабайта на 95%.
4. **Egress Traffic Minimization:** Передача даних між хмарами (Egress) — найдорожча стаття витрат у Multi-Cloud. Саме тому, як описано в Розділі 2, ми використовуємо жорсткий батчинг (Parquet + Snappy) для аналітики, а CDN Cloudflare бере на себе 60% вихідного трафіку, завдяки чому компанія не платить захмарні рахунки хмарним провайдерам за пропускну здатність.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **9. Практична симуляція (Chaos Engineering & Load Modeling)**

Теоретично бездоганна архітектура не гарантує надійності в умовах реального світу. Щоб переконатися, що **2BeMarket** витримає не лише пікові навантаження, але й каскадні збої інфраструктури, проект впроваджує практику хаос-інженерії та математичне моделювання процесів (Observability-Driven Development).

### 9.1. Практика Chaos Engineering (Парадокс стійкості)

Замість того, щоб чекати на збій, ми штучно інжектимо відмови в production-подібне середовище за допомогою інструментів Chaos Mesh. Це дозволяє перевірити, чи дійсно працюють механізми Graceful Degradation та автоматичного Failover.

Нижче наведено схему Game Day сценарію **"Деградація кешу"**, яка демонструє, як система рятує мобільний застосунок від "зависання" при падінні Redis.

```mermaid
graph TD
    MobileApp(("📱 Мобільний<br>застосунок"))

    subgraph Cluster ["☸️ Kubernetes Cluster"]
        Gateway["🌐 API Gateway"]
        OrderSvc["📦 Order Service<br>(Circuit Breaker)"]
        LocalCache[("⚡ Caffeine Cache<br>(Local In-Memory)")]
        Redis[("🔴 Redis Cluster<br>(Distributed)")]
        Chaos["🌪️ Chaos Mesh<br>(Fault Injector)"]
    end

    subgraph Observability ["📊 Telemetry (Prometheus + Grafana)"]
        Alerts((Alerts<br>Slack/PagerDuty))
    end

    MobileApp -->|GET /products| Gateway
    Gateway --> OrderSvc

    %% Normal flow
    OrderSvc -.->|Read| Redis

    %% Chaos Injection
    Chaos == "💉 Inject 2s Latency" ==> Redis

    %% Fallback flow
    OrderSvc == "🛡️ Timeout > 50ms<br>Fallback" ==> LocalCache

    %% Metrics
    Observability -.->|Spike Detected| Alerts
    OrderSvc -.->|Push Metrics| Observability

    classDef chaos fill:#3a0000,stroke:#ff0000,stroke-width:2px,color:#fff
    classDef safe fill:#004d40,stroke:#00bfa5,stroke-width:2px,color:#fff

    class Chaos chaos
    class Redis chaos
    class LocalCache safe
```

**Опис сценарію (Game Day):**
Ми штучно додаємо затримку у 2 секунди до відповідей Redis. Замість того, щоб чекати і "класти" клієнтські пули з'єднань, спрацьовує патерн **Circuit Breaker** (Запобіжник). `Order Service` миттєво відсікає Redis і переходить на читання локального In-Memory кешу.

*Результат:* Мобільна платформа продовжує отримувати каталог товарів за <50 мс, а інженери бачать графіки деградації у Grafana.

---

### 9.2. Data-Driven Архітектура: Візуалізація Asymmetric Scaling

Щоб довести інвесторам ефективність нашої стратегії оптимізації витрат (FinOps), ми змоделювали поведінку глобального трафіку під час регіонального розпродажу (Singles' Day в Азії). 

Наша математична модель (реалізована на Python + Plotly) наочно демонструє, що під час 5-кратного сплеску трафіку в кластері Alibaba Cloud (з базових ~27,000 до ~135,000 RPS), ресурси AWS (США) та GCP (Європа) залишаються на базовому рівні, не спалюючи бюджет проекту. 

Сумарний глобальний трафік у піку сягає рівно **175,000 RPS** (135k Азія + 15k Європа + 17k Америка + 8k MENA), ідеально торкаючись нашого жорсткого архітектурного ліміту. Система доводить здатність працювати на абсолютній межі заявлених потужностей без деградації.

In [69]:
# Симуляція даних - Створення анімації Plotly: Asymmetric Scaling (11.11)
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import base64
from IPython.display import HTML, display
import warnings

warnings.filterwarnings('ignore')

def simulate_global_traffic(intervals: int = 96, spike_start: int = 32, spike_end: int = 56) -> pd.DataFrame:
    np.random.seed(42)
    time_index = pd.date_range(start="2026-11-11 00:00", periods=intervals, freq="15min")

    eu_traffic, aws_traffic, azure_traffic, asia_traffic = [], [], [], []
    asia_pods, hpa_alerts = [], []

    for i in range(intervals):
        # Базове фонове навантаження у світі
        eu_gcp = int(max(10000, np.random.normal(15000, 1500)))       # ~15k RPS
        americas_aws = int(max(12000, np.random.normal(17000, 2000))) # ~17k RPS
        mena_azure = int(max(5000, np.random.normal(8000, 800)))      # ~8k RPS

        # Асиметричний сплеск в Азії (Singles' Day)
        if spike_start <= i <= spike_end:
            # Рівно 5-кратний сплеск (від ~27k до ~135k)
            # Сумарний глобальний трафік у піку: 135k + 15k + 17k + 8k = 175k RPS!
            asia_ali = int(np.random.uniform(130000, 140000)) 
        else:
            asia_ali = int(max(20000, np.random.normal(27000, 2000))) # База ~27k RPS

        eu_traffic.append(eu_gcp)
        aws_traffic.append(americas_aws)
        azure_traffic.append(mena_azure)
        asia_traffic.append(asia_ali)

        # Кожен Pod витримує 500 RPS
        pods = int(np.ceil(asia_ali / 500))
        asia_pods.append(pods)

        # Alerting якщо Азія пробиває поріг у 50k RPS (>100 Pods)
        if pods > 100 and (i == spike_start or i % 5 == 0):
            hpa_alerts.append(asia_ali)
        else:
            hpa_alerts.append(None)

    return pd.DataFrame({
        'Time': time_index,
        'GCP_RPS': eu_traffic,
        'AWS_RPS': aws_traffic,
        'Azure_RPS': azure_traffic,
        'Alibaba_RPS': asia_traffic,
        'Alibaba_Pods': asia_pods,
        'HPA_Alert': hpa_alerts
    })

df = simulate_global_traffic()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(x=df['Time'][:1], y=df['GCP_RPS'][:1], name="🇪🇺 GCP (Європа)", 
                         line=dict(color='#0F9D58', width=2), hovertemplate='%{y:,.0f} RPS'), secondary_y=False)
fig.add_trace(go.Scatter(x=df['Time'][:1], y=df['AWS_RPS'][:1], name="🌎 AWS (Америка/Сх.Азія)", 
                         line=dict(color='#FF9900', width=2), hovertemplate='%{y:,.0f} RPS'), secondary_y=False)
fig.add_trace(go.Scatter(x=df['Time'][:1], y=df['Azure_RPS'][:1], name="🌍 Azure (Африка/MENA/Австралія)", 
                         line=dict(color='#0078D4', width=2), hovertemplate='%{y:,.0f} RPS'), secondary_y=False)
fig.add_trace(go.Scatter(x=df['Time'][:1], y=df['Alibaba_RPS'][:1], name="🇨🇳 Alibaba (Китай/Пд-Сх.Азія)", 
                         line=dict(color='#ff3333', width=3), fill='tozeroy', fillcolor='rgba(255, 51, 51, 0.1)', hovertemplate='%{y:,.0f} RPS'), secondary_y=False)

fig.add_trace(go.Scatter(x=df['Time'][:1], y=df['Alibaba_Pods'][:1], name="📦 Alibaba Active Pods", 
                         line=dict(color='#00ffcc', width=2, dash='dot'), hovertemplate='%{y} Pods'), secondary_y=True)

fig.add_trace(go.Scatter(x=df['Time'][:1], y=df['HPA_Alert'][:1], mode='markers', name="⚠️ HPA Scale-Out",
                         marker=dict(color='yellow', size=12, symbol='triangle-up', line=dict(color='black', width=1)), hovertemplate='Scale-Out Triggered'), secondary_y=False)

frames = []
for i in range(1, len(df) + 1):
    frames.append(go.Frame(data=[
        go.Scatter(x=df['Time'][:i], y=df['GCP_RPS'][:i]),
        go.Scatter(x=df['Time'][:i], y=df['AWS_RPS'][:i]),
        go.Scatter(x=df['Time'][:i], y=df['Azure_RPS'][:i]),
        go.Scatter(x=df['Time'][:i], y=df['Alibaba_RPS'][:i]),
        go.Scatter(x=df['Time'][:i], y=df['Alibaba_Pods'][:i]),
        go.Scatter(x=df['Time'][:i], y=df['HPA_Alert'][:i])
    ], name=str(i)))
fig.frames = frames

animation_time_second = 15
frame_duration = int((animation_time_second * 1000) / len(df))

sliders = [dict(
    steps=[dict(method='animate', 
                args=[[str(i)], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))],
                label=df['Time'].dt.strftime('%H:%M').iloc[i-1]) for i in range(1, len(df) + 1)],
    active=0,
    transition=dict(duration=0),
    x=0, y=-0.05, 
    currentvalue=dict(font=dict(size=14, color="orange"), prefix="Час: ", visible=True, xanchor="right"),
    len=1.0
)]

fig.update_layout(
    title="<b>2BeMarket FinOps Dashboard:</b> Асиметричне масштабування 4-х хмар (11.11 Singles' Day)",
    height=950,
    template="plotly_dark",
    hovermode="x unified",
    hoverlabel=dict(namelength=-1), 
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    xaxis=dict(range=[df['Time'].min(), df['Time'].max()], title="Час доби"),
    yaxis=dict(range=[0, 180000], title="Запити за секунду (RPS)"),
    yaxis2=dict(range=[0, 500], title="Кількість активних Pods (HPA)", showgrid=False),
    sliders=sliders,
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        x=0.02, y=1.15,
        xanchor="left", yanchor="top",
        direction="right",
        pad={"r": 15, "t": 133},
        buttons=[
            dict(label=f"▶ Старт ({animation_time_second} сек)", method="animate",
                 args=[None, {"frame": {"duration": frame_duration, "redraw": True}, "fromcurrent": True}]),
            dict(label="⏸ Пауза", method="animate",
                 args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}])
        ]
    )]
)

fig.add_hline(y=50000, line_dash="dash", line_color="orange", 
              annotation_text="Початок масштабування (50k RPS)", 
              annotation_position="top left", secondary_y=False)

fig.add_hline(y=175000, line_dash="solid", line_color="red", 
              annotation_text="🚨 Абсолютний архітектурний ліміт системи (175k RPS)", 
              annotation_position="top left", secondary_y=False)

plot_html = fig.to_html(include_plotlyjs='cdn', full_html=True)
b64_chart = base64.b64encode(plot_html.encode()).decode()

download_link = f'<div style="text-align: center; margin-bottom: 10px;"><a href="data:text/html;base64,{b64_chart}" download="2bemarket_finops_dashboard.html" style="color:#a855f7; text-decoration: none; font-size: 16px; font-weight: bold; padding: 8px 16px; border: 1px solid #a855f7; border-radius: 5px; background-color: rgba(168, 85, 247, 0.1);">🛍️ Завантажити інтерактивний 2BeMarket</a></div>'
iframe_display = f'<iframe src="data:text/html;base64,{b64_chart}" width="100%" height="950px" frameborder="0"></iframe>'

display(HTML(download_link))
display(HTML(iframe_display))

> *Запуск цього коду генерує динамічний дашборд, який математично підтверджує, що HPA (Horizontal Pod Autoscaler) активується виключно в ураженому регіоні.*

---

### 9.3. Математична симуляція "Virtual Waiting Room" (Hype Drop на 1 Мільйон)

Окрім візуалізації трафіку, нам критично важливо гарантувати виживання транзакційного ядра під час Hype Drops (раптових запусків лімітованих товарів). Щоб наочно продемонструвати інвесторам ефективність патерну Edge-Queuing (*див. п. 5.6*), ми створили математичну модель на основі закону Літтла (Little's Law).

Уявімо реальний масштаб: **1,000,000 користувачів** одночасно натискають кнопку "Купити" у мобільному застосунку в першу секунду релізу.

Транзакційна база даних (CockroachDB/PostgreSQL) фізично здатна обробити лише **1,000 безпечних ACID-транзакцій на секунду**. Без CDN-черги база гарантовано ляже з помилкою *Connection Pool Exhausted*. 

**Нижче наведено Python-скрипт, який моделює роботу патерну "Зворотного тиску" (Backpressure) на рівні Edge-мережі:**

In [74]:
import time
import sys

def simulate_waiting_room(total_users=1000000, db_tps_limit=1000):
    print("🚀 СТАРТ СИМУЛЯЦІЇ: Hype Drop (Ексклюзивний реліз)")
    print(f"👥 Одночасний сплеск трафіку: {total_users:,} клієнтів")
    print(f"⚙️ Безпечний пропускний ліміт БД: {db_tps_limit:,} TPS")
    print("-" * 105)

    queue = total_users
    processed = 0
    seconds = 0
    bar_length = 40

    while queue > 0:
        seconds += 1
        batch = min(queue, db_tps_limit)

        queue -= batch
        processed += batch

        percent = processed / total_users
        filled = int(bar_length * percent)
        bar = '█' * filled + '-' * (bar_length - filled)

        sys.stdout.write(f"\r⏱️ {seconds:4d}s | [{bar}] {percent:5.1%} | Оброблено: {processed:9,} | В черзі: {queue:9,}")
        sys.stdout.flush()

        if seconds % 100 == 0:
            print() 

        time.sleep(0.002) 

    print("-" * 105)
    print(f"📊 Результат: 100% замовлень ({total_users:,}) успішно збережено. SLA підтримано на рівні 99.99%.")
    print(f"⏳ Максимальний час очікування для останнього клієнта: {(seconds/60):.1f} хвилин.")

if __name__ == "__main__":
    simulate_waiting_room()

🚀 СТАРТ СИМУЛЯЦІЇ: Hype Drop (Ексклюзивний реліз)
👥 Одночасний сплеск трафіку: 1,000,000 клієнтів
⚙️ Безпечний пропускний ліміт БД: 1,000 TPS
---------------------------------------------------------------------------------------------------------
⏱️  100s | [████------------------------------------] 10.0% | Оброблено:   100,000 | В черзі:   900,000
⏱️  200s | [████████--------------------------------] 20.0% | Оброблено:   200,000 | В черзі:   800,000
⏱️  300s | [████████████----------------------------] 30.0% | Оброблено:   300,000 | В черзі:   700,000
⏱️  400s | [████████████████------------------------] 40.0% | Оброблено:   400,000 | В черзі:   600,000
⏱️  500s | [████████████████████--------------------] 50.0% | Оброблено:   500,000 | В черзі:   500,000
⏱️  600s | [████████████████████████----------------] 60.0% | Оброблено:   600,000 | В черзі:   400,000
⏱️  700s | [████████████████████████████------------] 70.0% | Оброблено:   700,000 | В черзі:   300,000
⏱️  800s | [████████████

> Ми просимулювали роботу патерну "Зворотного тиску" (Backpressure) на рівні Edge-мережі, де система пропускає трафік батчами, що відповідають пропускній здатності БД.

**Результат виконання симуляції:** Архітектура довела свою фінансову та технічну спроможність. Замість того, щоб отримати каскадний збій через 1 мільйон одночасних запитів, база даних безпечно "перетравила" весь об'єм зі швидкістю 1,000 TPS за **~16.6 хвилин**, зберігаючи при цьому SLA на рівні 99.99%. 

**UX Висновок:** Замість фатальної помилки "503 Service Unavailable" або "мертвого" білого екрану, клієнти спостерігали прогнозовану живу чергу у мобільному застосунку. Архітектура конвертувала 100% хайпового трафіку в реальний прибуток проекту без жодного простою інфраструктури, ідеально балансуючи між агресивним масштабуванням та захистом капіталу (TCO - Total Cost of Ownership).

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **10. Загальні висновки (Executive Summary) та Вектор розвитку**

Проект **2BeMarket** являє собою еталонну Highload-систему, яка ідеально балансує між агресивним масштабуванням, суворою консистентністю даних та оптимізацією загальної вартості володіння (TCO - Total Cost of Ownership). 

### 10.1. Архітектурні досягнення та Бізнес-цінність (SLA 99.99%)

Цільовий показник доступності платформи розраховується за класичною формулою теорії надійності:

$$A = \frac{MTBF}{MTBF + MTTR} \ge 99.99\%$$

Це гарантує сумарний час простою не більше 52.6 хвилин на рік.

**Цей показник досягається завдяки синергії інженерних та бізнес-рішень:**

1. **Максимізація конверсії та Бездоганний UX:** Перенесення обчислень на рівень CDN (Embargo Caching та Virtual Waiting Room, *ефективність яких доведена симуляціями у Розділі 9*) кардинально знижує навантаження на БД. **Мобільний застосунок** та веб-портал залишаються швидкими (відгук < 50мс) і повністю інтерактивними навіть за умови 175,000 запитів на секунду. Користувачі ніколи не бачать помилок типу 503, що підвищує LTV (Life-Time Value) клієнта.
2. **Абсолютна консистентність та Захист капіталу (RPO = 0):** Використання патерну Saga Orchestration у поєднанні зі строгою прив'язкою до UTC гарантує, що жоден цент не "зависне" у повітрі. Geo-Partitioning та Quorum Consensus забезпечують нульову втрату даних навіть у разі розриву трансокеанічних кабелів (захист від Split-Brain).
3. **Еластичність та FinOps:** Концепція Asymmetric Scaling дозволяє платформі автоматично "здувати" інфраструктуру в періоди затишшя. Використання Spot-інстансів, ARM-процесорів та оптимізація Egress-трафіку економить до 40% бюджету.
    - *Asymmetric Scaling:* Під час локальних свят обчислювальні ресурси збільшуються виключно в ураженому регіоні, тоді як інші хмари залишаються на базовому рівні, заощаджуючи сотні тисяч доларів.
    - *Tail-based Sampling:* Інтелектуальний збір телеметрії економить до 90% бюджету на інфраструктуру логування, зберігаючи при цьому 100% критичної інформації для розслідування інцидентів.
4. **AI-Driven UX:** Впровадження GenAI (RAG) у якості Shopping Copilot для мобільного застосунку виводить взаємодію з каталогом на принципово новий рівень, а система розподіленого трасування (Jaeger) забезпечує повну прозорість усіх процесів.

---

### 10.2. Безкомпромісна безпека та Глобальний Комплаєнс (Security & AML)

Платформа готова до проходження найжорсткіших аудитів з боку регуляторів ЄС та США (GDPR, nFADP, PCI-DSS):

- Модель **Zero Trust** (mTLS, тимчасові секрети Vault) та патерн ABAC зводять радіус ураження до мінімуму навіть при компрометації внутрішнього периметра.
- Інтеграція **ML Risk Engine** у реальному часі відсікає спроби кардингу та фроду, захищаючи еквайрингові метрики платформи.
- Фіксування дій у **Immutable WORM S3** (незмінні аудит-логи) у поєднанні з автоматичним скринінгом за глобальними санкційними списками робить 2BeMarket юридично невразливою екосистемою.

---

### 10.3. Операційна досконалість та Швидкість інновацій (Operational Excellence)

Глобальний маркетплейс повинен адаптуватися до ринку щодня. Архітектура 2BeMarket знімає традиційний страх перед "п'ятничними релізами" і дозволяє випускати нові бізнес-фічі без зупинки платформи:

- **Оптимізований Клієнтський Досвід (BFF):** Використання патерну Backend for Frontend дозволяє ізольовано і швидко розвивати **мобільний застосунок** та Web-портал, гарантуючи, що клієнти на слабкому зв'язку отримують максимально "легкі" відповіді.
- **Безпечні глобальні релізи (Ring-based Deployments):** Завдяки кільцевому розгортанню та автоматизованому Rollback на основі метрик, бізнес може безпечно тестувати нові алгоритми на 5% аудиторії найменшого регіону. 
- **Проактивна стійкість (Chaos Engineering):** Платформа не чекає на "Чорну П'ятницю", щоб перевірити свою міцність. Регулярні симуляції збоїв (Game Days) гарантують, що автоматичний Failover спрацює саме тоді, коли на кону стоятимуть мільйони доларів прибутку.

---

### 10.4. Архітектурні компроміси та нюанси (Trade-offs)

Еталонна архітектура завжди вимагає свідомих компромісів. У проекті ми приймаємо наступні технічні трейд-офи:

- **Eventual Consistency (Кінцева узгодженість):** Заради високої пропускної здатності (High Availability за теоремою CAP), відображення статусів замовлень у **мобільному застосунку** може мати затримку до 2-3 секунд (час виконання кроків Saga).
- **Operational Complexity (Операційна складність):** Підтримка мультихмарної екосистеми (AWS, GCP, Azure, Alibaba) вимагає висококваліфікованої команди SRE та використання складних абстракцій (Crossplane / Terraform / OpenTelemetry).

---

### 10.5. Вектор майбутнього розвитку (Cell-based Architecture)

Забезпечивши поточні бізнес-потреби, архітектура проекту має чіткий вектор для подальшого гіпермасштабування. У разі виходу на нові ринки або досягнення позначки понад 500,000 RPS, система готова до переходу на **Cell-based Architecture (Стільникову архітектуру)**.

Замість того, щоб нескінченно збільшувати розмір глобальних кластерів (Scale-Up), проект розгортатиме ізольовані, самодостатні "стільники" (Cells) — міні-копії всієї платформи. Наприклад, "Cell 1" обслуговуватиме користувачів з ID від 1 до 10 млн.

Цей патерн дозволяє масштабуватися лінійно і математично зменшує радіус ураження (Blast Radius) за формулою:

$$R = \frac{1}{N}$$
*(де $N$ — кількість стільників).*

**Data Sovereignty (Суверенітет даних):** Окрім ізоляції збоїв, Cell-based Architecture вирішує геополітичну проблему. Наприклад, "Swiss Cell" може бути розгорнутий виключно в локальних ЦОД (AWS eu-central-2 у Цюриху) для повної відповідності вимогам nFADP, гарантуючи неможливість транскордонного витоку інформації.

---

**Фінальний підсумок:** 

Документ *Architecture Design Document (ADD)* засвідчує, що платформа **2BeMarket** має еталонний запас міцності. Вона здатна витримувати екстремальні хайп-дропи, миттєво адаптуватися до геополітичних чи інфраструктурних катастроф і забезпечувати безперебійний глобальний рух товарів та капіталу на роки вперед.

> ***Цей документ не просто описує набір технологій — він демонструє зріле інженерне мислення, готовність до вирішення глобальних бізнес-викликів, управління ризиками та здатність проектувати платформи світового класу.***

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **Додаток D: Детальний дизайн (Capacity, API & Data Models)**

Цей розділ містить низькорівневі розрахунки (Back-of-the-envelope), контракти взаємодії та схеми даних, які доводять математичну та структурну спроможність архітектури **2BeMarket** обслуговувати заявлені 10 млн DAU та 50+ млн товарів.

### D.1. Back-of-the-Envelope Розрахунки (Оцінка масштабу)

Для проектування місткості (Capacity Planning) використовуємо базові метрики з нефункціональних вимог:
- **DAU (Daily Active Users):** 10,000,000
- **Каталог товарів:** 50,000,000 активних SKU
- **Співвідношення Read/Write:** 100:1 (типово для E-commerce)

**1. Оцінка трафіку (RPS & Bandwidth):**
- *Середній трафік:* Якщо 1 користувач робить в середньому 50 запитів на день: 10,000,000 * 50 / 86400 ≈ 5,800 RPS (середнє базове навантаження).
- *Піковий трафік (Black Friday):* Задекларовано архітектурний ліміт у **175,000 RPS**.
- *Пропускна здатність (Egress Bandwidth):* При середньому розмірі відповіді 50 KB (JSON + оптимізовані WebP зображення), пікове мережеве навантаження складе: 175,000 * 50 KB ≈ 8.75 GB/s (або **~70 Gbps**). З них 70% візьме на себе CDN (Cloudflare), розвантажуючи власні сервери до безпечних 21 Gbps.

**2. Оцінка зберігання даних (Storage Capacity):**
- *Каталог товарів (ElasticSearch + MongoDB):* 50 млн товарів * 5 KB (метадані + текст) = **~250 GB** гарячих даних.
- *Медіафайли (S3 Object Storage):* 50 млн товарів * 5 фотографій * 200 KB = **~50 TB** (плюс відеоогляди = **~150 TB** загального обсягу).
- *Транзакції (Orders):* 100,000 транзакцій на годину в пік. При середньому показнику 500,000 замовлень на день * 2 KB (розмір JSON-замовлення) = **1 GB/день**. За 5 років зберігання транзакційне ядро (CockroachDB) накопичить лише **~1.8 TB** фінансових даних, що ідеально лягає в оперативну пам'ять кластера для надшвидких запитів.

---

### D.2. Дизайн API (REST & GraphQL Контракти)

Для комунікації між **мобільним застосунком**, Web-фронтендом та Backend-сервісами використовуються оптимізовані API-інтерфейси. Нижче наведено 4 ключові контракти.

**1. Аутентифікація користувача (Auth Service)**
- **Endpoint:** `POST /api/v1/auth/login`
- **Опис:** Перевіряє облікові дані та повертає короткостроковий JWT-токен.
- **Payload (Request):**
  ```json
  { "email": "user@mail.com", "password_hash": "a1b2c3d4...", "device_id": "uuid-1234" }
  ```
- **Response (200 OK):**
  ```json
  { "access_token": "jwt.header.payload.sig", "expires_in": 3600, "refresh_token": "secure_token" }
  ```

**2. Отримання деталей товару (Catalog Service - GraphQL)**
- **Endpoint:** `POST /graphql`
- **Опис:** GraphQL використовується для уникнення Over-fetching (завантаження зайвих даних у мобільному застосунку).
- **Query:**
  ```graphql
  query { product(id: "PROD-987") { name, price, stockStatus, reviews(limit: 5) { rating, text } } }
  ```

**3. Додавання товару до кошика (Cart Service)**
- **Endpoint:** `PUT /api/v1/cart/{user_id}/items`
- **Опис:** Ідемпотентний метод для оновлення стану кошика (зберігається в Redis/DynamoDB).
- **Payload (Request):**
  ```json
  { "product_id": "PROD-987", "quantity": 2, "idempotency_key": "uuid-9999" }
  ```
- **Response (200 OK):**
  ```json
  { "cart_total": 149.98, "item_count": 2, "status": "updated" }
  ```

**4. Ініціалізація Checkout (Order Service)**
- **Endpoint:** `POST /api/v1/checkout/process`
- **Опис:** Стартує розподілену транзакцію (Saga Orchestration) для списання коштів та резервування товару на складі.
- **Payload (Request):**
  ```json
  { "cart_id": "CART-123", "payment_method_id": "pm_card_567", "shipping_address_id": "ADDR-890" }
  ```
- **Response (202 Accepted):**
  ```json
  { "order_id": "ORD-555-777", "status": "PROCESSING", "saga_trace_id": "trace-abc-123" }
  ```

---

### D.3. Схеми баз даних (Data Models & Entities)

Через принцип Polyglot Persistence кожна сутність зберігається в оптимізованій для неї базі даних. Нижче наведено концептуальну ER-схему (Entity-Relationship) для 4-х ключових сутностей та опис їхніх структур.

```mermaid
erDiagram
    USERS ||--o{ ORDERS : "places"
	USERS ||--|| CARTS : "owns"
    USERS ||--o{ REVIEWS : "writes"
    ORDERS ||--|{ ORDER_ITEMS : "contains"
    PRODUCTS ||--o{ ORDER_ITEMS : "included_in"
    PRODUCTS ||--o{ REVIEWS : "receives"

    USERS {
        uuid id PK
        string email UK
        string password_hash
        jsonb preferences
        timestamp created_at
    }

    PRODUCTS {
        uuid id PK
        uuid merchant_id FK
        string name
        decimal price
        int stock_quantity
        jsonb attributes
    }

    ORDERS {
        uuid id PK
        uuid user_id FK
        decimal total_amount
        string status
        timestamp created_at
    }

    CARTS {
        uuid id PK
        uuid user_id FK "TTL-indexed"
        jsonb items "CRDT Vector"
        timestamp updated_at
    }
```

**Опис структур даних та вибір сховища:**

1. **Сутність `USERS` (CockroachDB):**
   - *Тип:* Реляційна (сувора консистентність).
   - *Особливості:* Поля з персональними даними (PII) зберігаються в зашифрованому вигляді. Поле `preferences` (JSONB) дозволяє гнучко додавати налаштування без зміни схеми (ALTER TABLE).
2. **Сутність `PRODUCTS` (ElasticSearch + MongoDB):**
   - *Тип:* Документо-орієнтована / Пошуковий індекс.
   - *Особливості:* Оскільки товари (смартфони, одяг) мають різні набори характеристик, реляційна таблиця створила б "розріджену" матрицю (Sparse Matrix). JSON-документи ідеально підходять для динамічного поля `attributes`.
3. **Сутність `CARTS` (Redis Enterprise):**
   - *Тип:* Key-Value (In-Memory або високошвидкісний диск).
   - *Особливості:* Кошик використовує структуру CRDT (Conflict-free Replicated Data Type) в полі `items` для уникнення втрати даних при одночасному додаванні товарів з Web-порталу та **мобільного застосунку**. Дані мають автоматичний TTL (Time-To-Live) на 30 днів для очищення "покинутих" кошиків.
4. **Сутність `ORDERS` (CockroachDB):**
   - *Тип:* Distributed NewSQL.
   - *Особливості:* Фінансове серце системи. Всі операції над цією таблицею є ACID-транзакціями. Статус (`PENDING`, `PAID`, `SHIPPED`) змінюється виключно через брокер повідомлень (Kafka) в рамках патерну Saga.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **Додаток E: Організаційна зрілість, Економіка та Екзистенційні Ризики (The Boardroom Chapter)**

Цей розділ призначений для C-level керівництва (CTO, CFO, CEO), інвесторів та Ради директорів. Він описує стратегію перетворення технічної архітектури **2BeMarket** на стійку, економічно вигідну та керовану бізнес-екосистему, здатну пережити інфраструктурні, геополітичні та організаційні кризи.

### E.1. Організаційна топологія (Закон Конвея)

Монолітна організація створює монолітний код. Для підтримки глобальної мікросервісної архітектури структура інженерних команд базується на принципах Team Topologies:

- **Stream-aligned teams:** Крос-функціональні команди, які повністю володіють своїм доменом. Наприклад, команда "Checkout" відповідає за `Order Service` від написання коду до підтримки в production.
- **Platform Engineering:** Виділена команда (SRE/DevOps), яка створює внутрішні інструменти (Internal Developer Platform). Їхня мета — надати командам розробки "асфальтовану дорогу" (Paved Road) для деплою, щоб продуктова команда, яка розробляє **мобільний застосунок**, могла випускати релізи щодня, не замислюючись про налаштування Kubernetes чи Istio.

---

### E.2. Unit-Економіка транзакції (Cloud FinOps)

Масштабування до 175,000 RPS має бути не лише технічно можливим, але й фінансово рентабельним. Платформа впроваджує жорсткий моніторинг Cloud Unit Economics:

- **Вартість кліка vs Вартість транзакції:** Система тегування хмарних ресурсів (AWS Cost Allocation Tags / GCP Labels) дозволяє вирахувати точну собівартість інфраструктури для кожної покупки. 
- Якщо маржинальність продажу становить $0.15, а мережевий трафік (Egress), читання з бази (CockroachDB) та виклик ML-моделей обходяться в $0.18, архітектура вважається збитковою. Asymmetric Scaling та Tail-based Sampling (див. попередні розділи) впроваджені саме для утримання Unit-економіки в позитивній зоні під час пікових навантажень.

---

### E.3. AI Governance та MLOps (Управління моделями)

Оскільки транзакційне ядро активно використовує ML Risk Engine для захисту від фроду та механізми рекомендацій, управління життєвим циклом штучного інтелекту стає критичним бізнес-фактором.

- **Запобігання Model Drift (Деградації моделі):** Поведінка шахраїв постійно еволюціонує. Архітектура передбачає автоматизовані MLOps пайплайни (наприклад, Kubeflow), які щотижня донавчають антифрод-моделі на нових патернах скасованих транзакцій.
- **AI Cost Control:** Запуск складних нейромереж для кожної транзакції може знищити бюджет. Тому важкі ML-моделі працюють асинхронно і кешують "Risk Score" користувача, щоб миттєві перевірки під час Checkout у **мобільному застосунку** виконувалися за <20 мс і коштували частки цента.

---

### E.4. Екологічна відповідальність (GreenOps та ESG)

Для європейського ринку відповідність критеріям ESG (Environmental, Social, and Governance) є обов'язковою умовою для залучення інституційних інвестицій.

- Архітектура 2BeMarket інтегрує інструменти GreenOps (наприклад, Cloud Carbon Footprint) для моніторингу вуглецевого сліду кластерів. 
- Балансувальники трафіку здатні динамічно маршрутизувати фонові, некритичні обчислення (наприклад, генерацію аналітичних звітів) у ті дата-центри, які в даний момент живляться на 100% від відновлюваних джерел енергії, знижуючи загальний рівень викидів CO2 (Carbon-Aware Routing).

---

### E.5. Сценарії "Чорного лебедя" (Vendor Exit Strategy)

Архітектура проектується з урахуванням подій з імовірністю 0.01%, які здатні знищити компанію (Extinction-Level Events):

- **Vendor Lock-in Doomsday:** Що відбудеться, якщо глобальний хмарний провайдер в односторонньому порядку розірве контракт або потрапить під санкції? Оскільки система запакована в Distroless контейнери і використовує відкриті стандарти (Kubernetes, Kafka, PostgreSQL-сумісні бази), RTO для повної міграції ядра на резервний кластер іншого провайдера становить менше 48 годин без переписування бізнес-логіки.
- **Постквантова криптографія (PQC):** Для захисту від майбутніх загроз з боку квантових комп'ютерів, здатних зламати поточні алгоритми (RSA/ECC), архітектура API Gateway та Service Mesh підтримує криптографічну гнучкість (Crypto-Agility), що дозволить безшовну міграцію на PQC-стандарти (Post-Quantum Cryptography) без оновлення коду мікросервісів.

---

### E.6. Культура реагування (Incident Command System)

Коли автоматичні RTO/RPO механізми не справляються, в гру вступає людський фактор. 

- Платформа використовує протокол **Incident Command System (ICS)**. У разі критичного збою (Severity 1) автоматично призначається Incident Commander, який бере на себе повне управління ситуацією, відключаючи бюрократію.
- **Blameless Post-Mortems (Бездефектний розбір польотів):** Якщо інженер випадково виконує деструктивну команду, що призводить до падіння продакшену, система розслідування фокусується не на покаранні людини, а на тому, *чому* архітектура та CI/CD пайплайн дозволили цій команді дійти до бази даних. Це створює культуру психологічної безпеки, де розробники проактивно повідомляють про архітектурні вразливості.

---

### E.7. M&A Readiness та B2B White-Labeling (Стратегія капіталізації)

Інфраструктура 2BeMarket спроектована не лише як B2C маркетплейс, але і як потенційний B2B SaaS-продукт:

- **Multi-Tenancy (White-Labeling):** Бази даних та API Gateway від самого початку підтримують мультитенантність (ізоляцію даних на рівні `tenant_id`). Це дозволяє компанії за лічені тижні розгорнути "клон" платформи для стороннього Enterprise-клієнта під його власним брендом.
- **M&A (Mergers and Acquisitions) Tech Due Diligence:** Завдяки строгій API-first архітектурі, Event-Driven дизайну (Kafka) та детальному Swagger/OpenAPI документуванню, компанія готова до швидкого поглинання менших конкурентів. Інтеграція придбаного стартапу в екосистему 2BeMarket зводиться до налаштування нових Kafka-конекторів та API-мостів, а не переписування монолітів.

---

### E.8. Total Cost of Ownership (TCO) та Структура OPEX (Скільки коштує це утримувати)

Для підтримання глобальної доступності (99.99%) та здатності обробляти 175,000 RPS, фінансова модель платформи переходить від капітальних інвестицій (CAPEX — купівля власних серверів) до гнучких операційних витрат (OPEX). 

Загальна вартість володіння (TCO) розподіляється за чотирма основними векторами:

1. **Хмарна Інфраструктура (Cloud & Network) — ~40% бюджету OPEX:**
    - **Compute & Storage:** Оплата кластерів Kubernetes (EKS/GKE) та кешу (Redis) у чотирьох хмарах. Найбільшою статтею витрат є транзакційна база даних (CockroachDB) через реплікацію SSD-дисків.
    - **Egress Traffic (Мережевий трафік):** Вихідний трафік (завантаження фото товарів, відео-оглядів у **мобільному застосунку**) є прихованою пасткою хмар. Використання Cloudflare як глобального CDN знижує витрати на Egress AWS/GCP до 70%.
2. **Ліцензії, SaaS та Обсервабільність — ~15% бюджету OPEX:**
    - Витрати на телеметрію (Jaeger/Elasticsearch) для зберігання петабайтів логів (зменшено завдяки Tail-based Sampling).
    - Enterprise-ліцензії: GitHub Enterprise, HashiCorp Vault, інструменти безпеки (WAF, Sift Fraud Engine).
3. **Фонд Оплати Праці (Payroll & R&D) — ~40% бюджету OPEX:**
    - **Інженерія:** Зарплати крос-функціональних команд (iOS/Android розробники для **мобільного застосунку**, Backend, Frontend), Platform/SRE інженерів (чергування 24/7), QA та Data-сайєнтистів (ML-моделі).
    - **Операційна підтримка:** Комплаєнс-офіцери (розгляд заморожених KYC/AML транзакцій), служба клієнтської підтримки, менеджери по роботі з ключовими мерчантами (Key Account Managers).
4. **Юридичні витрати та Комплаєнс — ~5% бюджету OPEX:**
    - Щорічний аудит PCI-DSS, послуги зовнішніх DPO (Data Protection Officers) для GDPR-комплаєнсу в Європі, підписки на бази даних санкцій (LexisNexis).

---

### E.9. Модель монетизації, ROI та Unit-Економіка (Скільки це принесе)

Високі інфраструктурні витрати повністю компенсуються гіпермасштабованою моделлю монетизації. Мета архітектури — знизити вартість обробки однієї транзакції (Cost per Order) до часток цента, максимізуючи чисту маржу.

Платформа генерує дохід (Revenue) через три незалежні потоки:

1. **Комісійна модель (Take-Rate / Transaction Fees):** - Базовий дохід. Платформа утримує від 5% до 15% з кожної успішної транзакції (залежно від категорії товару). 
    - *Архітектурна цінність:* Патерни "Waiting Room" та Saga Orchestration гарантують, що під час Hype Drop на 100,000 товарів по $1,000 кожен (GMV = $100M), система не впаде. Навіть при комісії 5%, архітектура гарантовано приносить **$5,000,000 чистого доходу за 100 секунд** роботи.
2. **FinTech та Платіжні послуги:**
    - Платформа заробляє на арбітражі еквайрингу. Завдяки Smart Payment Routing, транзакції динамічно направляються через найдешевший локальний шлюз. Різниця між комісією, яку платить покупець, та реальною комісією еквайєра залишається в компанії.
3. **Retail Media Network (Рекламна мережа B2B):**
    - Найбільш високомаржинальний (до 80%) продукт. Продавці платять за пріоритетну видачу товарів у каталозі (Sponsored Products) або банери у **мобільному застосунку**. 
    - *Архітектурна цінність:* Швидкість завантаження каталогу (<50мс завдяки BFF та Redis) напряму впливає на Click-Through Rate (CTR) рекламних блоків. Що швидше працює API, то більше кліків роблять користувачі, і то більше грошей продавців спалюється на користь платформи.

**Висновок щодо ROI (Return on Investment):**
Високі початкові витрати на розгортання Multi-Cloud Active-Active архітектури є обґрунтованими. Математична модель доводить: вартість 1 години простою (Downtime) під час "Чорної П'ятниці" у 10 разів перевищує річну вартість резервного кластера в Alibaba Cloud. Інвестиції в надійність безпосередньо захищають Revenue платформи.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **Додаток X: Classified Operations & State-Level Security (The Black Book)**

*УВАГА: Цей розділ призначений виключно для ознайомлення Радою Директорів (Board of Directors), CEO, CTO та CISO. Документ описує протоколи реагування на екзистенційні загрози, втручання на рівні держав (State-Sponsored Actors) та механізми захисту від глобальних інфраструктурних катастроф.*

Стандартні архітектурні патерни (mTLS, WAF, ACID) захищають платформу від кіберзлочинців та технічних збоїв. Проте глобальна екосистема **2BeMarket** проектується з урахуванням ризиків іншого порядку: геополітичних конфліктів, санкцій, банкрутства хмарних провайдерів та фізичного шпигунства. 

Для нейтралізації цих загроз впроваджено 7 протоколів найвищого рівня доступу:

### X.1. Confidential Computing (Апаратна ізоляція оперативної пам'яті)

Традиційна криптографія захищає дані в дорозі (mTLS) та на диску (KMS / WORM S3). Проте в момент обробки транзакції (наприклад, валідації PII або розрахунку кошика) дані знаходяться у відкритому вигляді в оперативній пам'яті (RAM) сервера. У звичайній хмарі адміністратор AWS або GCP з фізичним доступом до сервера гіпервізора може зробити дамп пам'яті та викрасти ключі або персональні дані.

**Протокол захисту:** Транзакційне ядро `Order Service` та сервіси обробки персональних даних розгортаються виключно на базі **Confidential VMs (GCP)** або **AWS Nitro Enclaves**. Ці технології використовують апаратне шифрування оперативної пам'яті на рівні процесора (наприклад, AMD SEV-SNP).

- *Бізнес-імпакт:* Навіть маючи root-доступ до фізичного сервера або ухвалу суду на зняття дампів пам'яті, ні співробітники хмарного провайдера, ні спецслужби не зможуть прочитати дані клієнтів. Це забезпечує ультимативну відповідність вимогам швейцарського FADP та європейського GDPR.

---

### X.2. Geopolitical "Kill Switch" (Процедура "Розбитого скла")

У разі миттєвого введення міжнародних санкцій (рівня OFAC), початку військових дій або критичної компрометації цілого державного сегмента мережі, класичний процес відкату коду (CI/CD Rollback) є занадто повільним. Платформа потребує можливості миттєвої ампутації ураженого сегмента.

**Протокол захисту:** Впроваджено апаратний та конфігураційний "Kill Switch" на двох рівнях:

1. *External (BGP / DNS):* На рівні Cloudflare Incident Commander може за 1 секунду перевести цілий макрорегіон (наприклад, весь близькосхідний або азійський кластер) у режим "Чорної діри" (Blackhole routing), миттєво припинивши прийом будь-якого трафіку.
2. *Internal (Service Mesh):* Крос-хмарні приватні канали зв'язку (VPC Peering / Cloud Interconnect) між ураженим регіоном та глобальним Data Lake миттєво розриваються на рівні Istio. Це створює "карантин", який унеможливлює юридичну чи вірусну контагіозність (зараження інших кластерів), дозволяючи решті світу продовжувати генерувати прибуток.

---

### X.3. Traffic Mirroring (Симуляція реального "Судного дня")

Синтетичні навантажувальні тести (Chaos Mesh, k6) ніколи не відтворюють 100% реальної поведінки ботнетів, фроду та аномалій під час екстремальних навантажень (175,000 RPS). Тестування нових архітектурних рішень безпосередньо на реальних розпродажах несе неприпустимий ризик для Revenue.

**Протокол захисту:** Використання патерну **Shadow Traffic (Тіньовий трафік)**. API Gateway / Istio налаштовані таким чином, що можуть асинхронно і непомітно *дублювати* 100% живого, реального трафіку (з реальними JWT-токенами та кошиками) і відправляти його на прихований, повністю ізольований **Dark Cluster**.

- Цей кластер обробляє справжні транзакції користувачів (з імітацією відповідей від платіжних шлюзів), дозволяючи інженерам спостерігати, як нова база даних або логіка "вмирає" під справжнім штормом запитів, тоді як реальні покупці в **мобільному застосунку** не відчувають жодної затримки і продовжують покупки на стабільному Production-кластері.

---

### X.4. Code & Data Escrow (Сховище "Останньої надії")

Найбільший прихований ризик мультихмарної архітектури — це глобальний Vendor Lock-in на рівні інфраструктури керування або одночасне банкрутство/відключення ключових провайдерів через глобальну кібервійну чи економічну кризу. Якщо AWS та GCP заблокують корпоративні акаунти, бізнес перестане існувати.

**Протокол захисту:** Для забезпечення абсолютної незалежності компанії укладено угоду про **Software & Data Escrow**.

- Раз на добу автоматичний пайплайн бере зашифровані зліпки (Snapshots) транзакційного ядра CockroachDB, весь інфраструктурний код (Terraform/Crossplane), Docker-образи ключових мікросервісів та кореневі криптографічні ключі (KMS). 
- Ці дані вивантажуються на фізичні накопичувачі WORM-типу, які зберігаються в автономних офлайн-бункерах найвищого рівня безпеки (наприклад, у швейцарських Альпах — Swiss Fort Knox). Це гарантує, що навіть у разі повного "стирання" цифрового сліду компанії в публічних хмарах, бізнес **2BeMarket** можна буде легально та фізично "воскресити" на приватному "голому залізі" (Bare Metal) протягом кількох діб.

---

### X.5. Протокол "Level Zero": Autonomous AI Defense та Ghost Sharding

Стандартні системи захисту (WAF, Rate Limiting) розраховані на блокування відомих векторів атак та ботнетів. Проте у випадку цілеспрямованої атаки урядового рівня (Advanced Persistent Threat — APT), пряме блокування є антипатерном, оскільки воно сигналізує зловмиснику про те, що його викрито, змушуючи змінити тактику на більш приховану.

**Протокол захисту (Honeypot-роутинг у реальному часі):** Замість блокування, система використовує концепцію **Ghost Sharding**.

1. Спеціалізований AI-агент безперервно моніторить системні виклики ядра Linux (через eBPF) на всіх вузлах транзакційного кластера.
2. Як тільки фіксується мікроаномалія (наприклад, спроба ін'єкції нульового дня або нетиповий дамп пам'яті), алгоритм за мілісекунди "клонує" цільовий мікросервіс і піднімає ізольовану "тіньову" базу даних (Ghost DB), заповнену синтетичними, згенерованими LLM даними, що імітують реальні фінансові потоки.
3. Service Mesh (наприклад, Istio) миттєво, без розриву TCP-з'єднання, перемикає хакера на цей "Ghost Cluster". 
Зловмисник годинами продовжує атаку, вважаючи, що викрадає реальні транзакції, але насправді він ізольований і лише тренує нашу ML-модель своїми найсучаснішими векторами. У цей час реальні користувачі, які роблять покупки через **мобільний застосунок**, навіть не підозрюють про атаку, оскільки їхній трафік маршрутизується в безпечній зоні.

---

### X.6. Протокол "Plausible Deniability": BYOK 2.0 (Bring Your Own Key)

Незважаючи на апаратну ізоляцію серверів (Confidential Computing), найслабшою ланкою залишається ризик інсайдерської загрози (наприклад, компрометація адміністратора бази даних) або офіційний юридичний примус з боку авторитарних держав до розкриття історії транзакцій конкретного користувача.

**Протокол захисту (Апаратна ізоляція на рівні клієнта):**

Платформа впроваджує архітектуру *Cryptographic Plausible Deniability* (Криптографічне правдоподібне заперечення). 

1. Майстер-ключ шифрування PII-даних та історії покупок генерується і зберігається виключно в апаратному модулі (Secure Enclave / Titan M) безпосередньо у смартфоні клієнта.
2. Серверна база даних зберігає інформацію клієнта у вигляді абсолютно нечитабельного шифротексту і фізично не має ключів для його дешифрування.
3. Коли клієнт відкриває **мобільний застосунок**, пристрій криптографічно підписує короткостроковий мандат (наприклад, з TTL 10 хвилин) і передає його в захищений анклав сервера. 

Тільки на час дії цього мандата бекенд здатний розшифрувати дані в оперативній пам'яті для відображення списку замовлень. Як тільки клієнт закриває застосунок або мандат спливає, дані знову перетворюються на криптографічний шум. У разі будь-якого зовнішнього тиску на платформу, компанія гарантовано захищена математикою: вона фізично не здатна надати дані у відкритому вигляді без прямої та добровільної участі пристрою самого користувача.

---

### X.7. Протокол "Scorched Earth" (Криптографічне самознищення при фізичному захопленні)

Усі цифрові протоколи є вразливими, якщо ворожі спецслужби або військові здійснюють раптове фізичне захоплення локального дата-центру (Data Center Raid). Використовуючи фізичну ізоляцію від магістральних інтернет-каналів (щоб унеможливити активацію X.2 "Kill Switch") та атаки типу "Cold Boot" (заморожування серверів рідким азотом для зняття дампів RAM), нападники можуть спробувати витягти майстер-ключі з працюючих машин.

**Протокол захисту (Апаратний "Dead Man's Switch"):** Платформа впроваджує процедуру **Crypto-Shredding** (Криптографічне знищення) на базі апаратних модулів безпеки (HSM) із сертифікацією FIPS 140-3 Level 4 (вищий рівень захисту від несанкціонованого втручання).

1. **Автономна детонація:** Алгоритм не чекає команди від людей. Якщо сервери транзакційного ядра втрачають високочастотний "Heartbeat" (криптографічний пульс) з глобальним кворумом на інших континентах більше ніж на 3 секунди (ознака примусового відключення від глобальної мережі), або якщо фізичні сенсори шасі (датчики світла, тиску, температури) фіксують спробу розкриття стійки.
2. **Zeroization (Обнулення):** HSM-модуль миттєво ініціює апаратне стирання. Локальні генератори ентропії за мілісекунди перезаписують усі кореневі ключі шифрування (KMS) випадковим шумом на рівні кремнієвого чіпа. 
3. **Ефект ентропії:** У цю ж мить петабайти даних на всіх локальних SSD-масивах (включно з базами даних користувачів та транзакціями) незворотно перетворюються на беззмістовну математичну ентропію (цифрове сміття).

**Бізнес-імпакт:** Нападники отримують під контроль сотні тонн дорогого "мертвого металу", але рівно 0 байт персональних чи фінансових даних клієнтів. Користувачі **мобільного застосунку** в інших макрорегіонах навіть не відчують простою, оскільки трафік буде миттєво перенаправлено, а знищений кластер буде згодом відновлено на безпечній території з "Чистих" бекапів (Протокол X.4). Це гарантує, що 2BeMarket ніколи не стане інструментом у руках авторитарних режимів.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #000000, #434343, #000000); border-radius: 2px;">